
# BARAM 2026 AutoGluon Baseline

This notebook keeps the current preprocessing and train-only SHAP feature-selection flow, but replaces manual grid search with AutoGluon Tabular.

Flow:

1. Install packages and import libraries.
2. Define preprocessing, feature-selection, and AutoGluon settings.
3. Load the datasets.
4. Run the feature engineering flow from `BARAM2026_preprocess_reference.ipynb`.
5. Select features using LightGBM feature importance and SHAP values.
6. Train group-wise AutoGluon regime models.
7. Diagnose validation residuals and AutoGluon leaderboards.
8. Predict 2025 and save a submission file.



## Experiment Note

- Feature selection: final top `2,000` features per group.
- Manual grid search is removed.
- Model I zero classifier uses one fixed LightGBM parameter set.
- Model II capacity classifier and Model III regressor use AutoGluon with `GBM` and `XGB` only.
- The regression component uses MAE/L1 evaluation with optional sample weights.



## (1) Imports

Import libraries only. All paths, preprocessing options, model parameters, and runtime switches are defined in the next configuration cell.


In [ ]:

# Package setup for Colab/local execution.
import importlib.util
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_package("lightgbm")
ensure_package("xgboost")
ensure_package("autogluon.tabular", "autogluon.tabular")
ensure_package("shap")
ensure_package("seaborn")
ensure_package("sklearn", "scikit-learn")


In [ ]:

from itertools import combinations
from pathlib import Path

import gc
import hashlib
import itertools
import json
import random
import shutil
import time
import warnings

import numpy as np
import pandas as pd

import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor
from autogluon.tabular import TabularPredictor

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as e:
    SHAP_AVAILABLE = False
    print(f"[WARN] SHAP unavailable: {e}")

warnings.filterwarnings(
    "ignore",
    message=".*LightGBM binary classifier with TreeExplainer shap values output has changed.*",
    category=UserWarning,
)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOT_AVAILABLE = True
except Exception as e:
    PLOT_AVAILABLE = False
    print(f"[WARN] Plotting unavailable: {e}")

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    drive = None



## (2) Configuration

This cell contains every experiment setting used later in the notebook.

- **Paths and outputs** define where train/test files are loaded and where artifacts are saved.
- **Preprocessing settings** control weather aggregation, spatial grids, calendar features, lag features, and rolling features.
- **Feature selection settings** control the LightGBM importance + SHAP feature filter.
- **AutoGluon settings** control the automated model selection, hyperparameter tuning, and weighted ensemble.


In [ ]:
# =============================
# Display and runtime basics
# =============================

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


# =============================
# Paths
# =============================

if IN_COLAB:
    drive.mount("/content/drive")

COLAB_DATA_DIR = Path("/content/drive/MyDrive/BARAM2026/data")
LOCAL_DATA_DIR = Path(r"C:\Users\심현석\Documents\BARAM 2026")

if IN_COLAB:
    BASE_DIR = COLAB_DATA_DIR
elif LOCAL_DATA_DIR.exists():
    BASE_DIR = LOCAL_DATA_DIR
else:
    BASE_DIR = Path.cwd()

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESS_EXPERIMENT_NAME = "preprocess_reference"
FEATURE_OUTPUT_DIR = OUTPUT_DIR / f"{PREPROCESS_EXPERIMENT_NAME}_features"
FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELING_EXPERIMENT_NAME = "autogluon_baseline"
EXP_DIR = OUTPUT_DIR / MODELING_EXPERIMENT_NAME
CHECKPOINT_DIR = EXP_DIR / "checkpoints"
PLOT_DIR = EXP_DIR / "plots"
IMPORTANCE_DIR = EXP_DIR / "importance"
SHAP_DIR = EXP_DIR / "shap"
FEATURE_SELECTION_DIR = EXP_DIR / "feature_selection"
AUTOGLUON_MODEL_DIR = EXP_DIR / "autogluon_models"
for path_ in [EXP_DIR, CHECKPOINT_DIR, PLOT_DIR, IMPORTANCE_DIR, SHAP_DIR, FEATURE_SELECTION_DIR, AUTOGLUON_MODEL_DIR]:
    path_.mkdir(parents=True, exist_ok=True)

AUTOGLUON_VALID_SUMMARY_PATH = CHECKPOINT_DIR / "autogluon_validation_summary.csv"
AUTOGLUON_LEADERBOARD_PATH = CHECKPOINT_DIR / "autogluon_leaderboard.csv"
BEST_SUMMARY_PATH = CHECKPOINT_DIR / "autogluon_best_summary.csv"
VALID_PRED_PATH = CHECKPOINT_DIR / "validation_predictions.csv"
VALID_METRIC_PATH = CHECKPOINT_DIR / "validation_metrics.csv"
RESIDUAL_LONG_PATH = CHECKPOINT_DIR / "validation_residuals_long.csv"
WRONG_CASE_PATH = CHECKPOINT_DIR / "wrong_regime_cases_with_features.csv"
SUBMISSION_PATH = EXP_DIR / "baseline_submit.csv"
FINAL_SUMMARY_PATH = CHECKPOINT_DIR / "final_model_summary.csv"
FEATURE_IMPORTANCE_PATH = IMPORTANCE_DIR / "lgbm_builtin_feature_importance.csv"
PERMUTATION_IMPORTANCE_PATH = IMPORTANCE_DIR / "permutation_importance.csv"
SHAP_IMPORTANCE_PATH = SHAP_DIR / "shap_importance.csv"
FEATURE_SELECTION_SUMMARY_PATH = FEATURE_SELECTION_DIR / "feature_selection_summary.csv"
FEATURE_SELECTION_COMBINED_PATH = FEATURE_SELECTION_DIR / "combined_shap_top_features.csv"


# =============================
# Competition constants
# =============================

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21_600.0,
    "kpx_group_2": 21_600.0,
    "kpx_group_3": 21_000.0,
}


# =============================
# Preprocessing settings
# =============================

WEATHER_AGGS = ["mean", "std", "max"]
SUBSET_AGGS = ["mean", "std", "max"]

# Convert LDAPS precipitation into binary flags.
# The 15mm+ bin is treated as the reference bin and is not explicitly created.
PRECIP_SOURCE_COL = "surface_0_ncpcp"
PRECIP_BIN_FEATURES = [
    ("precip_0_1mm", 0.0, 1.0),
    ("precip_1_3mm", 1.0, 3.0),
    ("precip_3_15mm", 3.0, 15.0),
]

# Control whether binary flag columns are included in spatial aggregations.
COUNT_REGIME_FLAGS_IN_AGGREGATIONS = False
COUNT_PRECIP_FLAGS_IN_AGGREGATIONS = False

# Control whether every LDAPS/GFS grid is summarized with all-grid aggregations.
INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS = True
INCLUDE_GFS_ALL_GRID_AGGREGATIONS = True

# LDAPS raw grids used directly by each target group.
LDAPS_GROUP_RAW_GRID_IDS = {
    "kpx_group_1": [5, 6],
    "kpx_group_2": [6, 11],
    "kpx_group_3": [6, 12],
}

# LDAPS grid subsets used for group-specific subset aggregation.
LDAPS_GROUP_AGGREGATION_REFERENCE_GRIDS = {
    "kpx_group_1": [[5, 6, 10, 11]],
    "kpx_group_2": [[6, 7, 11, 12]],
    "kpx_group_3": [[6, 11, 12, 16]],
}

# Combination sizes generated from each reference grid list.
# The default [4] keeps only the full 4-grid subset from each reference list.
LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES = [4]

# lag/rolling feature parameters.
WEATHER_LAGS = [1, 2, 24]
WEATHER_ROLL_WINDOWS = [3, 6, 24]


# =============================
# Validation and regime settings
# =============================

VALIDATION_OPTION = "full_2024"

USE_ENSEMBLE_OPTION = False
ENSEMBLE_WEIGHTS = {"lgbm_regime": 1.0}

ZERO_THRESHOLD = 0.80
CAPACITY_THRESHOLD = 0.90
ZERO_LABEL_KWH = 100.0
CAPACITY_LABEL_RATIO = 0.98

# Model I is intentionally kept simple and deterministic.
ZERO_CLASSIFIER_PARAMS = {
    "n_estimators": 800,
    "learning_rate": 0.035,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}



# =============================
# Runtime switches
# =============================

RUN_AUTOGLUON_TRAIN = True
FORCE_RERUN_AUTOGLUON = False
RUN_FINAL_TRAIN_AND_SUBMIT = True
RUN_DIAGNOSTICS = True

# AutoGluon exposes its own leaderboard and optional permutation importance.
# Built-in LightGBM feature importance / SHAP diagnostics are not used for the final AutoGluon predictors.
RUN_AUTOGLUON_FEATURE_IMPORTANCE = False

RUN_SHAP_FEATURE_SELECTION = True
REUSE_FEATURE_SELECTION_CACHE = True
FORCE_RERUN_FEATURE_SELECTION = False
ALLOW_LEGACY_FEATURE_SELECTION_CACHE = True


# =============================
# Feature selection and diagnostics
# =============================

# Feature selection is two-stage:
# 1) LightGBM feature importance top-K shortlist
# 2) SHAP top-K final feature set
FEATURE_SELECTION_TOPK_IMPORTANCE = 5000
FEATURE_SELECTION_TOPK_SHAP = 2000
FEATURE_SELECTION_SHAP_SAMPLE_SIZE = 2000
FEATURE_SELECTION_MAX_ESTIMATORS = 800

FEATURE_SELECTION_CLASSIFIER_PARAMS = {
    "n_estimators": 800,
    "learning_rate": 0.035,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}

FEATURE_SELECTION_REGRESSOR_PARAMS = {
    "n_estimators": 1200,
    "learning_rate": 0.02,
    "num_leaves": 127,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}

CASE_EXPORT_MAX_FEATURES = 120
PERMUTATION_SAMPLE_SIZE = 2000
PERMUTATION_N_REPEATS = 3


# =============================
# Sample-weight settings
# =============================

USE_POWER_SAMPLE_WEIGHT = True
MIN_NON_EVAL_WEIGHT = 0.35
HIGH_POWER_WEIGHT_STRENGTH = 6.0


# =============================
# AutoGluon settings
# =============================

AG_LABEL_COL = "__label__"
AG_SAMPLE_WEIGHT_COL = "__sample_weight__"

AUTOGLUON_PRESETS = "medium_quality"
AUTOGLUON_TIME_LIMIT_PER_COMPONENT = None
AUTOGLUON_NUM_TRIALS = 12
AUTOGLUON_NUM_BAG_FOLDS = 0
AUTOGLUON_NUM_STACK_LEVELS = 0

# Set this to 1 in a GPU runtime if you want AutoGluon to pass GPU resources
# to supported models. XGBoost can use GPU; LightGBM GPU support depends
# a GPU-enabled LightGBM build.
AUTOGLUON_NUM_GPUS = 0
AUTOGLUON_VERBOSITY = 2
AUTOGLUON_WEIGHT_EVALUATION = False

AUTOGLUON_HYPERPARAMETERS = {
    "GBM": {},
    "XGB": {},
}

AUTOGLUON_HYPERPARAMETER_TUNE_KWARGS = {
    "num_trials": AUTOGLUON_NUM_TRIALS,
    "scheduler": "local",
    "searcher": "auto",
}


print("BASE_DIR:", BASE_DIR)
print("FEATURE_OUTPUT_DIR:", FEATURE_OUTPUT_DIR)
print("Experiment directory:", EXP_DIR)
print("AutoGluon model directory:", AUTOGLUON_MODEL_DIR)
print("RUN_AUTOGLUON_TRAIN:", RUN_AUTOGLUON_TRAIN)
print("FORCE_RERUN_AUTOGLUON:", FORCE_RERUN_AUTOGLUON)
print("RUN_SHAP_FEATURE_SELECTION:", RUN_SHAP_FEATURE_SELECTION)
print("REUSE_FEATURE_SELECTION_CACHE:", REUSE_FEATURE_SELECTION_CACHE)
print("FORCE_RERUN_FEATURE_SELECTION:", FORCE_RERUN_FEATURE_SELECTION)
print("FEATURE_SELECTION_TOPK_IMPORTANCE:", FEATURE_SELECTION_TOPK_IMPORTANCE)
print("FEATURE_SELECTION_TOPK_SHAP:", FEATURE_SELECTION_TOPK_SHAP)
print("AUTOGLUON_HYPERPARAMETERS:", AUTOGLUON_HYPERPARAMETERS)
print("AUTOGLUON_HYPERPARAMETER_TUNE_KWARGS:", AUTOGLUON_HYPERPARAMETER_TUNE_KWARGS)
print("AUTOGLUON_NUM_GPUS:", AUTOGLUON_NUM_GPUS)
print("ZERO_CLASSIFIER_PARAMS:", ZERO_CLASSIFIER_PARAMS)
print("USE_POWER_SAMPLE_WEIGHT:", USE_POWER_SAMPLE_WEIGHT)


## (3) Load Data

- 데이터 불러오기
- train_labels에서 2022년 그룹 3의 발전량은 NaN으로 두기


In [ ]:
required_paths = [
    TRAIN_DIR / "train_labels.csv",
    TRAIN_DIR / "ldaps_train.csv",
    TRAIN_DIR / "gfs_train.csv",
    TEST_DIR / "ldaps_test.csv",
    TEST_DIR / "gfs_test.csv",
    BASE_DIR / "sample_submission.csv",
]

labels_raw = pd.read_csv(TRAIN_DIR / "train_labels.csv")
labels_raw["kst_dtm"] = pd.to_datetime(labels_raw["kst_dtm"])
labels_raw = labels_raw.sort_values("kst_dtm").reset_index(drop=True)

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv")
sample_submission = pd.read_csv(BASE_DIR / "sample_submission.csv")

for df in [ldaps_train, gfs_train, ldaps_test, gfs_test]:
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    df["data_available_kst_dtm"] = pd.to_datetime(df["data_available_kst_dtm"])

sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("labels:", labels_raw.shape)
print("ldaps_train/test:", ldaps_train.shape, ldaps_test.shape)
print("gfs_train/test:", gfs_train.shape, gfs_test.shape)
print("sample_submission:", sample_submission.shape)

In [ ]:
def count_missing_values(labels: pd.DataFrame) -> pd.DataFrame:
    out = labels.sort_values("kst_dtm").copy()
    out["kst_dtm"] = pd.to_datetime(out["kst_dtm"])

    cutoff = pd.Timestamp("2023-01-01 00:00:00")
    audit_rows = []

    for target in TARGET_COLS:
        missing_mask = out[target].isna()
        if target == "kpx_group_3":
            count = int((missing_mask & (out["kst_dtm"] > cutoff)).sum())
        else:
            count = int(missing_mask.sum())
        audit_rows.append({"kpx_group": target, "count": count})

    return pd.DataFrame(audit_rows)

display(count_missing_values(labels_raw))

Check missing group-3 labels after 2023-01-01.


In [ ]:
labels_raw[labels_raw['kpx_group_3'].isna() & (labels_raw["kst_dtm"] > pd.Timestamp("2023-01-01 00:00:00"))]

## (4) Feature Engineering
- 물리적으로 말이 안 되는 값 보정
- 일부 기상변수 제거
- 파생변수 정의

### Drop Columns
- 일부 기상변수 제거


In [ ]:
LDAPS_DROP_COLS = [
    "surface_0_NDNSW",
    "surface_0_NDNLW",
    "heightAboveGround_2_SWDIR",
    "heightAboveGround_2_SWDIF",
    "etc_0_hcc",
    "etc_0_mcc",
    "etc_0_lcc",
    "etc_0_VLCDC",
    "surface_0_avg_lsprate",
    "surface_0_lssrate",
    "meanSea_0_prmsl",
    # "surface_0_ncpcp",
    "surface_0_snol",
    "surface_0_SNOM",
    "surface_0_lsm",
    "surface_0_h",
]

GFS_DROP_COLS = [
    "meanSea_0_prmsl",
    "surface_0_dswrf",
    "surface_0_dlwrf",
    "surface_0_prate",
    "surface_0_tp",
    "lowCloudLayer_0_lcc",
    "middleCloudLayer_0_mcc",
    "highCloudLayer_0_hcc",
    "atmosphere_0_tcc",
]

def drop_existing_columns(df: pd.DataFrame, cols: list[str], name: str) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    if existing:
        df = df.drop(columns=existing)
    print(f"{name}: dropped {len(existing)} columns | remaining columns={df.shape[1]}")
    return df

ldaps_train = drop_existing_columns(ldaps_train, LDAPS_DROP_COLS, "ldaps_train")
ldaps_test = drop_existing_columns(ldaps_test, LDAPS_DROP_COLS, "ldaps_test")
gfs_train = drop_existing_columns(gfs_train, GFS_DROP_COLS, "gfs_train")
gfs_test = drop_existing_columns(gfs_test, GFS_DROP_COLS, "gfs_test")


### Physical Value Corrections

Clip physically impossible humidity and dew-point values after dropping unused columns.


In [ ]:
def fix_physical_weather_values(
    df: pd.DataFrame,
    name: str,
    rh_col: str,
    dewpoint_col: str,
    temp_col: str,
) -> pd.DataFrame:
    out = df.copy()

    # Relative humidity cannot exceed 100%.
    rh_fixed = 0
    if rh_col in out.columns:
        rh_fixed = int((out[rh_col] > 100).sum())
        out[rh_col] = out[rh_col].clip(upper=100)

    # Dew point cannot be higher than air temperature.
    dewpoint_fixed = 0
    if dewpoint_col in out.columns and temp_col in out.columns:
        mask = out[dewpoint_col] > out[temp_col]
        dewpoint_fixed = int(mask.sum())
        out.loc[mask, dewpoint_col] = out.loc[mask, temp_col]

    print(f"{name}: clipped RH rows={rh_fixed:,} | corrected dewpoint rows={dewpoint_fixed:,}")
    return out


ldaps_train = fix_physical_weather_values(
    ldaps_train,
    "ldaps_train",
    rh_col="heightAboveGround_2_r",
    dewpoint_col="heightAboveGround_2_dpt",
    temp_col="heightAboveGround_2_t",
)
ldaps_test = fix_physical_weather_values(
    ldaps_test,
    "ldaps_test",
    rh_col="heightAboveGround_2_r",
    dewpoint_col="heightAboveGround_2_dpt",
    temp_col="heightAboveGround_2_t",
)
gfs_train = fix_physical_weather_values(
    gfs_train,
    "gfs_train",
    rh_col="heightAboveGround_2_2r",
    dewpoint_col="heightAboveGround_2_2d",
    temp_col="heightAboveGround_2_2t",
)
gfs_test = fix_physical_weather_values(
    gfs_test,
    "gfs_test",
    rh_col="heightAboveGround_2_2r",
    dewpoint_col="heightAboveGround_2_2d",
    temp_col="heightAboveGround_2_2t",
)


#### Precipitation Distribution Check

Check the distribution of `surface_0_ncpcp` in `ldaps_train` using the 0~1mm, 1~3mm, 3~15mm, and 15mm+ bins, ignoring `grid_id`.


In [ ]:
precip_col = PRECIP_SOURCE_COL
precip_bins = [0, 1, 3, 15, np.inf]
precip_labels = ["0~1mm", "1~3mm", "3~15mm", "15mm~"]

ldaps_precip = ldaps_train.copy()
ldaps_precip["forecast_kst_dtm"] = pd.to_datetime(ldaps_precip["forecast_kst_dtm"])

ldaps_precip["precip_bin"] = pd.cut(
    ldaps_precip[precip_col].astype(float).clip(lower=0),
    bins=precip_bins,
    labels=precip_labels,
    right=False,
    include_lowest=True,
)

precip_train_overall = (
    ldaps_precip["precip_bin"]
    .value_counts()
    .reindex(precip_labels, fill_value=0)
    .rename_axis("precip_bin")
    .reset_index(name="count")
)
precip_train_overall["ratio"] = precip_train_overall["count"] / precip_train_overall["count"].sum()
precip_train_overall["ratio_pct"] = precip_train_overall["ratio"] * 100
display(precip_train_overall)

### Physics-Feature Engineering Helpers

1. 물리 기반 파생변수 생성 함수 정의
   - 풍속(sqrt(u^2+v^2)), 풍속^2, 풍속^3, 풍속^(-2), 풍속^(-3)
   - sin(풍향), cos(풍향)
   - 공기 밀도
   - power_proxy: 0.5 * 공기 밀도 * swept area * 유효풍속^3 <br><br>

2. 그룹별로 참조하는 grid의 범위를 정의


In [ ]:
LDAPS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_50_50MUmax", "heightAboveGround_50_50MVmax", "wind_50m_max"),
    ("heightAboveGround_50_50MUmin", "heightAboveGround_50_50MVmin", "wind_50m_min"),
    ("heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS", "xbl_wind_5m"),
]

GFS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "wind_10m"),
    ("heightAboveGround_80_u", "heightAboveGround_80_v", "wind_80m"),
    ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "wind_100m"),
    ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl_wind"),
    ("isobaricInhPa_850_u", "isobaricInhPa_850_v", "wind_850hpa"),
    ("isobaricInhPa_700_u", "isobaricInhPa_700_v", "wind_700hpa"),
    ("isobaricInhPa_500_u", "isobaricInhPa_500_v", "wind_500hpa"),
]

LDAPS_SPEED_COLS = [f"{name}_speed" for _, _, name in LDAPS_UV_PAIRS]
GFS_SPEED_COLS = [f"{name}_speed" for _, _, name in GFS_UV_PAIRS]

GROUP_REGIME_SPECS = {
    "kpx_group_1": {"alias": "g1", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_2": {"alias": "g2", "cut_in": 3.0, "rated": 12.0, "cut_out": 22.5},
    "kpx_group_3": {"alias": "g3", "cut_in": 3.0, "rated": 12.5, "cut_out": 22.0},
}


# Build a readable subset name such as g1_s_5_6_10_11.
def subset_name(alias: str, grid_ids: list[int]) -> str:
    return f"{alias}_s_{'_'.join(map(str, grid_ids))}"


# Generate aggregation subsets from the requested combination sizes.
def combination_subsets(alias: str, grid_ids: list[int], sizes: list[int] | tuple[int, ...]) -> dict[str, list[int]]:
    subsets = {}
    for size in sizes:
        if size <= 1 or size > len(grid_ids):
            continue
        for combo in combinations(grid_ids, size):
            combo_ids = list(combo)
            subsets[subset_name(alias, combo_ids)] = combo_ids
    return subsets


# Merge aggregation subsets from one or more reference grid groups.
def merge_combination_subsets(
    alias: str,
    grid_groups: list[list[int]],
    sizes: list[int] | tuple[int, ...],
) -> dict[str, list[int]]:
    subsets = {}
    for grid_ids in grid_groups:
        subsets.update(combination_subsets(alias, grid_ids, sizes))
    return subsets


# Build the final LDAPS spatial plan from the configuration cell.
def build_ldaps_spatial_plan(
    raw_grid_ids: dict[str, list[int]],
    aggregation_reference_grids: dict[str, list[list[int]]],
    combination_sizes: list[int] | tuple[int, ...],
) -> dict[str, dict[str, object]]:
    plan = {}
    for target in TARGET_COLS:
        alias = GROUP_REGIME_SPECS[target]["alias"]
        plan[target] = {
            "raw_grid_ids": raw_grid_ids.get(target, []),
            "subsets": merge_combination_subsets(
                alias,
                aggregation_reference_grids.get(target, []),
                combination_sizes,
            ),
        }
    return plan


LDAPS_GROUP_SPATIAL_PLAN = build_ldaps_spatial_plan(
    LDAPS_GROUP_RAW_GRID_IDS,
    LDAPS_GROUP_AGGREGATION_REFERENCE_GRIDS,
    LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES,
)

GFS_RAW_GRID_IDS = [3, 5]


# Calculate air density by using weather features.
def safe_inverse(values: pd.Series, power: int = 1) -> pd.Series:
    vals = values.astype(float)
    return pd.Series(np.where(vals > 0, 1.0 / np.power(vals, power), np.nan), index=values.index)

def add_air_density_features(out: pd.DataFrame) -> pd.DataFrame:
    temp_col = next((c for c in ["heightAboveGround_2_t", "heightAboveGround_2_2t"] if c in out.columns), None)
    humidity_col = next((c for c in ["heightAboveGround_2_q", "heightAboveGround_2_2sh"] if c in out.columns), None)
    pressure_col = "surface_0_sp" if "surface_0_sp" in out.columns else None
    if temp_col is None or humidity_col is None or pressure_col is None:
        return out

    temp_k = out[temp_col].astype(float).clip(lower=150.0, upper=360.0)
    specific_humidity = out[humidity_col].astype(float).clip(lower=0.0, upper=0.08)
    pressure_pa = out[pressure_col].astype(float).clip(lower=50_000.0, upper=110_000.0)

    # 공기 밀도는 습도를 고려하는 공식으로 고정
    out["humid_air_density"] = pressure_pa / (287.05 * temp_k * (1.0 + 0.61 * specific_humidity))
    out["humid_air_density_inv"] = safe_inverse(out["humid_air_density"], power=1)

    return out


# Add physics features by using wind features
def add_wind_physics_features(df: pd.DataFrame, uv_pairs: list[tuple[str, str, str]], target: str) -> pd.DataFrame:
    out = df.copy()
    speed_cols = []
    for u_col, v_col, name in uv_pairs:
        if u_col not in out.columns or v_col not in out.columns:
            continue

        u = out[u_col].astype(float)
        v = out[v_col].astype(float)
        speed = np.sqrt(u * u + v * v)
        angle = np.arctan2(v, u)
        speed_col = f"{name}_speed"

        out[speed_col] = speed
        out[f"{name}_speed_sq"] = speed ** 2
        out[f"{name}_speed_cubed"] = speed ** 3
        out[f"{name}_speed_pow_neg2"] = safe_inverse(speed, power=2)
        out[f"{name}_speed_pow_neg3"] = safe_inverse(speed, power=3)
        out[f"{name}_angle_sin"] = np.sin(angle)
        out[f"{name}_angle_cos"] = np.cos(angle)
        speed_cols.append(speed_col)

    out = add_air_density_features(out)

    spec = GROUP_REGIME_SPECS[target]
    rotor_radius_m = 68.0 if target == "kpx_group_3" else 63.0
    swept_area = np.pi * rotor_radius_m * rotor_radius_m

    for speed_col in speed_cols:
        raw_speed = out[speed_col].astype(float)
        effective_speed = raw_speed.copy()
        effective_speed = effective_speed.mask(raw_speed < spec["cut_in"], 0.0)
        effective_speed = effective_speed.mask(raw_speed >= spec["cut_out"], 0.0)
        effective_speed = effective_speed.mask((raw_speed >= spec["rated"]) & (raw_speed < spec["cut_out"]), spec["rated"])

        out[f"{speed_col}_power_proxy"] = 0.5 * out["humid_air_density"] * swept_area * effective_speed ** 3

    return out


### Flag and Spatial Aggregation Helpers

- 풍속을 구간화하는 0-1 변수 생성 함수 (regime flag)
  - 0 ~ cutin / cutin ~ rated / rated ~ cutout kWh
- 강수량을 구간화하는 0-1 변수 생성 함수 (precipitation_bin_flag)
  - 0 ~ 1 / 1 ~ 3 / 3 ~ 15 mm
- 원본 데이터는 하나의 예보 대상 시각에 대해 16개 or 9개의 행이 존재하므로, 각 grid별 기상 예보값을 하나의 행에 합쳐놓는 함수
- 각 예보 대상 시각별로, 모든/일부 grid에 대해 기상 예보값을 요약하는 파생변수 생성 함수 (mean, std, max)


In [ ]:
# Create binary wind-regime flags for each available wind-speed column.
def add_group_speed_regime_flags(grid_df: pd.DataFrame, target: str, speed_cols: list[str]) -> pd.DataFrame:
    out = grid_df.copy()

    # Load the cut-in, rated, and cut-out speeds for the target group.
    spec = GROUP_REGIME_SPECS[target]
    cut_in = spec["cut_in"]
    rated = spec["rated"]
    cut_out = spec["cut_out"]

    # Create one flag for each wind-speed regime.
    for speed_col in speed_cols:
        if speed_col not in out.columns:
            continue
        speed = out[speed_col].astype(float)
        out[f"{speed_col}_below_cutin"] = (speed < cut_in).astype(float)
        out[f"{speed_col}_cutin_to_rated"] = ((speed >= cut_in) & (speed < rated)).astype(float)
        out[f"{speed_col}_rated_to_cutout"] = ((speed >= rated) & (speed < cut_out)).astype(float)

    return out


# Return columns created by add_group_speed_regime_flags.
def regime_flag_cols(df: pd.DataFrame) -> list[str]:
    suffixes = ("_below_cutin", "_cutin_to_rated", "_rated_to_cutout")
    return [c for c in df.columns if c.endswith(suffixes)]


# Return precipitation-bin flag columns created by add_precipitation_bin_flags.
def precip_flag_cols(df: pd.DataFrame) -> list[str]:
    names = [name for name, _, _ in PRECIP_BIN_FEATURES]
    return [c for c in names if c in df.columns]


# Convert raw precipitation into binary bin flags. The 15mm+ bin is kept as the omitted baseline.
def add_precipitation_bin_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if PRECIP_SOURCE_COL not in out.columns:
        return out

    precip = out[PRECIP_SOURCE_COL].astype(float).clip(lower=0)
    for name, lower, upper in PRECIP_BIN_FEATURES:
        out[name] = ((precip >= lower) & (precip < upper)).astype(float)

    return out


# Select numeric weather feature columns while excluding identifiers, timestamps, locations, and raw u/v components.
def numeric_weather_cols(
    df: pd.DataFrame,
    exclude_regime_flags: bool = False,
    exclude_precip_flags: bool = False,
) -> list[str]:
    # Exclude the raw precipitation column because only the precipitation-bin flags should reach the final data.
    skip = {"forecast_id", "forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude", PRECIP_SOURCE_COL}
    uv_component_cols = {col for u_col, v_col, _ in [*LDAPS_UV_PAIRS, *GFS_UV_PAIRS] for col in (u_col, v_col)}
    skip.update(uv_component_cols)
    cols = [c for c in df.columns if c not in skip and pd.api.types.is_numeric_dtype(df[c])]

    # Optionally remove wind-regime flags from numeric aggregation bases.
    if exclude_regime_flags:
        flag_set = set(regime_flag_cols(df))
        cols = [c for c in cols if c not in flag_set]

    # Optionally remove precipitation-bin flags from numeric aggregation bases.
    if exclude_precip_flags:
        flag_set = set(precip_flag_cols(df))
        cols = [c for c in cols if c not in flag_set]

    return cols


# Pivot selected grid IDs into wide columns without aggregating them.
def raw_grid_wide(grid_df: pd.DataFrame, prefix: str, grid_ids: list[int]) -> pd.DataFrame:
    # Start from one row per forecast time.
    out = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    cols = numeric_weather_cols(grid_df, exclude_regime_flags=False, exclude_precip_flags=False)

    # Add one set of feature columns for each selected grid_id.
    for gid in grid_ids:
        sub = grid_df.loc[grid_df["grid_id"] == gid, ["forecast_kst_dtm", *cols]].copy()
        rename = {c: f"{prefix}_g{gid:02d}_{c}" for c in cols}
        out = out.merge(sub.rename(columns=rename), on="forecast_kst_dtm", how="left")

    return out


# Aggregate all available grids into one row per forecast time.
def aggregate_all_grids(grid_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    # Select base numeric columns for mean/std/min/max aggregation.
    time_index = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    base_cols = numeric_weather_cols(grid_df, exclude_regime_flags=True, exclude_precip_flags=True)
    flag_cols = regime_flag_cols(grid_df)
    precip_cols = precip_flag_cols(grid_df)

    # Store each aggregation block before merging them together.
    pieces = []

    # Aggregate ordinary numeric weather features across all grids.
    if base_cols:
        agg = grid_df.groupby("forecast_kst_dtm", sort=True)[base_cols].agg(WEATHER_AGGS)
        agg.columns = [f"{prefix}_all_{col}_{stat}" for col, stat in agg.columns]
        pieces.append(agg.reset_index())

    # Optionally count how many grids fall into each wind-speed regime.
    if COUNT_REGIME_FLAGS_IN_AGGREGATIONS and flag_cols:
        counts = grid_df.groupby("forecast_kst_dtm", sort=True)[flag_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_all_{c}_count" for c in flag_cols})
        pieces.append(counts)

    # Optionally count how many grids fall into each precipitation bin. The 15mm+ baseline is not counted.
    if COUNT_PRECIP_FLAGS_IN_AGGREGATIONS and precip_cols:
        counts = grid_df.groupby("forecast_kst_dtm", sort=True)[precip_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_all_{c}_count" for c in precip_cols})
        pieces.append(counts)

    # Merge all aggregation blocks back onto the forecast-time index.
    out = time_index
    for frame in pieces:
        out = out.merge(frame, on="forecast_kst_dtm", how="left")

    return out


# Aggregate one selected subset of grid IDs into one row per forecast time.
def aggregate_subset(grid_df: pd.DataFrame, prefix: str, subset_name: str, grid_ids: list[int]) -> pd.DataFrame:
    # Select base numeric columns for mean/std/min/max aggregation.
    time_index = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    base_cols = numeric_weather_cols(grid_df, exclude_regime_flags=True, exclude_precip_flags=True)
    flag_cols = regime_flag_cols(grid_df)
    precip_cols = precip_flag_cols(grid_df)
    selected_cols = ["forecast_kst_dtm", "grid_id", *base_cols, *flag_cols, *precip_cols]
    sub = grid_df.loc[grid_df["grid_id"].isin(grid_ids), selected_cols].copy()
    if sub.empty:
        return time_index

    # Store each aggregation block before merging them together.
    pieces = []

    # Aggregate ordinary numeric weather features across the selected grids.
    if base_cols:
        agg = sub.groupby("forecast_kst_dtm", sort=True)[base_cols].agg(SUBSET_AGGS)
        agg.columns = [f"{prefix}_{subset_name}_{col}_{stat}" for col, stat in agg.columns]
        pieces.append(agg.reset_index())

    # Optionally count how many selected grids fall into each wind-speed regime.
    if COUNT_REGIME_FLAGS_IN_AGGREGATIONS and flag_cols:
        counts = sub.groupby("forecast_kst_dtm", sort=True)[flag_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_{subset_name}_{c}_count" for c in flag_cols})
        pieces.append(counts)

    # Optionally count how many selected grids fall into each precipitation bin. The 15mm+ baseline is not counted.
    if COUNT_PRECIP_FLAGS_IN_AGGREGATIONS and precip_cols:
        counts = sub.groupby("forecast_kst_dtm", sort=True)[precip_cols].sum(numeric_only=True).reset_index()
        counts = counts.rename(columns={c: f"{prefix}_{subset_name}_{c}_count" for c in precip_cols})
        pieces.append(counts)

    # Merge all aggregation blocks back onto the forecast-time index.
    out = time_index
    for frame in pieces:
        out = out.merge(frame, on="forecast_kst_dtm", how="left")

    return out

### Calendar, Lag and Rolling Helpers

1. 시간 파생변수 생성 함수 정의
   - hour, day, month, season
   - 각각 sin, cos 변환
   - 2024년은 윤년임을 고려하여 변환함 <br><br>

2. lag, rolling 파생변수 생성 함수 정의


In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out["forecast_kst_dtm"])

    hour = dt.dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    day = dt.dt.dayofyear
    days_in_year = np.where(dt.dt.is_leap_year, 366, 365)
    out["day_sin"] = np.sin(2 * np.pi * day / days_in_year)
    out["day_cos"] = np.cos(2 * np.pi * day / days_in_year)

    month = dt.dt.month
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)

    season = ((dt.dt.month % 12) // 3).astype(int)
    out["season_sin"] = np.sin(2 * np.pi * season / 4)
    out["season_cos"] = np.cos(2 * np.pi * season / 4)

    return out


def choose_lag_roll_cols(df: pd.DataFrame, subset_names: list[str], max_cols: int | None = None) -> list[str]:
    # Use the global limit only when it exists and the caller did not pass a limit.
    if max_cols is None:
        max_cols = globals().get("MAX_LAG_ROLL_BASE_COLS", None)

    weather_keywords = [
        "wind", "speed", "gust", "power_proxy", "precip", "pressure_diff",
        "blh", "planetaryBoundaryLayer", "VRATE",
        "humid_air_density", "surface_0_sp",
        "heightAboveGround_2_t", "heightAboveGround_2_2t",
        "heightAboveGround_2_q", "heightAboveGround_2_2sh",
    ]
    regime_keywords = ["below_cutin", "cutin_to_rated", "rated_to_cutout"]

    # 이 컬럼이 raw_grid_wide()에서 나온 컬럼인지 확인
    def is_raw_grid_col(col: str) -> bool:
        parts = col.split("_")
        return len(parts) >= 3 and len(parts[1]) == 3 and parts[1].startswith("g") and parts[1][1:].isdigit()

    # 이 컬럼이 현재 그룹의 subset aggregation에서 나온 컬럼인지 확인
    def is_subset_col(col: str) -> bool:
        return any(subset_name in col for subset_name in subset_names)

    # 이미 만들어진 lag/rolling 컬럼인지 확인
    def is_existing_lag_roll_col(col: str) -> bool:
        return "_lag" in col or "_roll" in col

    # 수치형 변수만 골라내기
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    # Keep numeric weather, regime, raw-grid, and subset columns as possible lag/rolling bases.
    candidates = [
        col
        for col in numeric_cols
        if not is_existing_lag_roll_col(col)
        and (
            any(keyword in col for keyword in weather_keywords)
            or any(keyword in col for keyword in regime_keywords)
            or is_raw_grid_col(col)
            or is_subset_col(col)
        )
    ]

    # 개수 제한이 없거나, 후보 개수가 제한보다 작으면 그대로 반환
    if max_cols is None or len(candidates) <= max_cols:
        return candidates

    ##################### 이 아래는 후보가 너무 많을 때만 실행됨 #####################

    # subset 관련 컬럼은 무조건 포함하도록 함
    subset_cols = [col for col in candidates if is_subset_col(col)]

    # raw grid, 공기 밀도, power_proxy, regime flag 컬럼은 무조건 포함하도록 함
    protected_cols = [
        col
        for col in candidates
        if is_raw_grid_col(col)
        or "humid_air_density" in col
        or "surface_0_sp" in col
        or "pressure_diff" in col
        or "blh" in col
        or "planetaryBoundaryLayer" in col
        or "VRATE" in col
        or "speed_pow_neg" in col
        or "power_proxy" in col
        or "precip" in col
        or any(keyword in col for keyword in regime_keywords)
    ]

    # subset_cols와 protected_cols를 합치되 중복은 제거
    protected = list(dict.fromkeys(subset_cols + protected_cols))

    # Return all protected columns when protected columns alone exceed the requested limit.
    if len(protected) > max_cols:
        selected = protected
        excluded = [col for col in candidates if col not in selected]
        choose_lag_roll_cols.last_excluded_cols = excluded
        print(f"[WARN] Protected columns ({len(protected)}) exceed max_cols ({max_cols}). Return all protected columns.")
        return selected

    # 후보 컬럼들의 분산을 계산.
    # 분산이 큰 컬럼은 시간에 따라 값이 많이 변하는 컬럼이고, lag/rolling feature를 만들었을 때 정보량이 있을 가능성이 더 크다고 보기.
    variances = df[candidates].var(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # 분산이 큰 순서대로 컬럼 정렬, 이미 보호된 컬럼은 제외. max_cols개까지만 선택.
    ranked = [col for col in variances.sort_values(ascending=False).index.tolist() if col not in protected]
    selected = (protected + ranked)[:max_cols]
    excluded = [col for col in candidates if col not in selected]
    choose_lag_roll_cols.last_excluded_cols = excluded

    return selected


def add_lag_rolling_features(df: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    out = df.sort_values("forecast_kst_dtm").reset_index(drop=True).copy()
    new_features = {}

    for col in base_cols:
        s = out[col].astype(float)

        # Lag 파생변수 생성 (1, 2, 3, 24)
        for lag in WEATHER_LAGS:
            new_features[f"{col}_lag{lag}"] = s.shift(lag)
        shifted = s.shift(1)

        # Rolling 파생변수 생성 (3, 6, 12, 24)
        for window in WEATHER_ROLL_WINDOWS:
            new_features[f"{col}_roll{window}_mean"] = shifted.rolling(window, min_periods=1).mean()
            new_features[f"{col}_roll{window}_std"] = shifted.rolling(window, min_periods=2).std()

    if new_features:
        out = pd.concat([out, pd.DataFrame(new_features, index=out.index)], axis=1)

    return out


def merge_feature_frames(frames: list[pd.DataFrame]) -> pd.DataFrame:
    non_empty = [f for f in frames if f is not None and len(f.columns) > 1]
    if not non_empty:
        raise ValueError("No feature frames to merge.")
    out = non_empty[0]
    for frame in non_empty[1:]:
        out = out.merge(frame, on="forecast_kst_dtm", how="outer")
    return out


### Build Group-Specific Feature Frames

#### 파생변수를 포함한 그룹별 데이터 생성, feature selection 전

Combine calendar features, optional all-grid LDAPS/GFS aggregations, group-specific LDAPS raw grid features, and selected subset aggregation features. Raw LDAPS grids, aggregation reference grids, precipitation flag counts, all-grid aggregation toggles, and aggregation combination sizes are controlled in the configuration cell.


In [ ]:
started = time.time()

FEATURE_INVENTORY_PATH = FEATURE_OUTPUT_DIR / "feature_inventory.csv"
labels = labels_raw.copy()


def add_ldaps_blh_diff_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create per-grid BLH first and second differences along forecast time."""
    if "etc_0_blh" not in df.columns:
        return df

    out = df.sort_values(["grid_id", "forecast_kst_dtm"]).reset_index(drop=True).copy()
    blh = out["etc_0_blh"].astype(float)
    grouped = blh.groupby(out["grid_id"], sort=False)
    out["etc_0_blh_diff1"] = grouped.diff(1)
    out["etc_0_blh_diff2"] = grouped.diff(2)
    return out


def prepare_weather_grid(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.copy()
    out["forecast_kst_dtm"] = pd.to_datetime(out["forecast_kst_dtm"])
    out["data_available_kst_dtm"] = pd.to_datetime(out["data_available_kst_dtm"])
    out = out.sort_values(["forecast_kst_dtm", "grid_id", "data_available_kst_dtm"])
    out = out.groupby(["forecast_kst_dtm", "grid_id"], as_index=False).tail(1).reset_index(drop=True)
    if prefix == "ldaps":
        out = add_ldaps_blh_diff_features(out)
    out = add_precipitation_bin_flags(out)
    return out


def add_target_wind_physics_features(df: pd.DataFrame, uv_pairs: list[tuple[str, str, str]], target: str) -> pd.DataFrame:
    out = df.copy()
    speed_cols = []

    for u_col, v_col, name in uv_pairs:
        if u_col not in out.columns or v_col not in out.columns:
            continue
        u = out[u_col].astype(float)
        v = out[v_col].astype(float)
        speed = np.sqrt(u * u + v * v)
        angle = np.arctan2(v, u)
        speed_col = f"{name}_speed"

        out[speed_col] = speed
        out[f"{name}_speed_sq"] = speed ** 2
        out[f"{name}_speed_cubed"] = speed ** 3
        out[f"{name}_speed_pow_neg2"] = safe_inverse(speed, power=2)
        out[f"{name}_speed_pow_neg3"] = safe_inverse(speed, power=3)
        out[f"{name}_angle_sin"] = np.sin(angle)
        out[f"{name}_angle_cos"] = np.cos(angle)
        speed_cols.append(speed_col)

    temp_col = next((c for c in ["heightAboveGround_2_t", "heightAboveGround_2_2t"] if c in out.columns), None)
    humidity_col = next((c for c in ["heightAboveGround_2_q", "heightAboveGround_2_2sh"] if c in out.columns), None)
    pressure_col = "surface_0_sp" if "surface_0_sp" in out.columns else None

    if temp_col is not None and humidity_col is not None and pressure_col is not None:
        temp_k = out[temp_col].astype(float).clip(lower=150.0, upper=360.0)
        specific_humidity = out[humidity_col].astype(float).clip(lower=0.0, upper=0.08)
        pressure_pa = out[pressure_col].astype(float).clip(lower=50_000.0, upper=110_000.0)
        out["humid_air_density"] = pressure_pa / (287.05 * temp_k * (1.0 + 0.61 * specific_humidity))
        out["humid_air_density_inv"] = safe_inverse(out["humid_air_density"], power=1)

        spec = GROUP_REGIME_SPECS[target]
        rotor_radius_m = 68.0 if target == "kpx_group_3" else 63.0
        swept_area = np.pi * rotor_radius_m * rotor_radius_m

        for speed_col in speed_cols:
            raw_speed = out[speed_col].astype(float)
            effective_speed = raw_speed.copy()
            effective_speed = effective_speed.mask(raw_speed < spec["cut_in"], 0.0)
            effective_speed = effective_speed.mask(raw_speed >= spec["cut_out"], 0.0)
            effective_speed = effective_speed.mask((raw_speed >= spec["rated"]) & (raw_speed < spec["cut_out"]), spec["rated"])
            out[f"{speed_col}_power_proxy"] = 0.5 * out["humid_air_density"] * swept_area * effective_speed ** 3

    return out


def gfs_grid_surface_pressure_diff(
    grid_df: pd.DataFrame,
    high_grid_id: int = 3,
    low_grid_id: int = 5,
) -> pd.DataFrame:
    """Create GFS surface-pressure difference: grid_3 pressure minus grid_5 pressure."""
    out = grid_df[["forecast_kst_dtm"]].drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
    feature_name = f"gfs_g{high_grid_id:02d}_minus_g{low_grid_id:02d}_surface_0_sp_pressure_diff"

    required = {"forecast_kst_dtm", "grid_id", "surface_0_sp"}
    if not required.issubset(grid_df.columns):
        out[feature_name] = np.nan
        return out

    sub = grid_df.loc[
        grid_df["grid_id"].isin([high_grid_id, low_grid_id]),
        ["forecast_kst_dtm", "grid_id", "surface_0_sp"],
    ].copy()
    if sub.empty:
        out[feature_name] = np.nan
        return out

    wide = sub.pivot_table(
        index="forecast_kst_dtm",
        columns="grid_id",
        values="surface_0_sp",
        aggfunc="mean",
    )

    if high_grid_id in wide.columns and low_grid_id in wide.columns:
        diff = (wide[high_grid_id] - wide[low_grid_id]).rename(feature_name).reset_index()
        out = out.merge(diff, on="forecast_kst_dtm", how="left")
    else:
        out[feature_name] = np.nan

    return out


ldaps_all = pd.concat([ldaps_train, ldaps_test], ignore_index=True)
gfs_all = pd.concat([gfs_train, gfs_test], ignore_index=True)

print("Preparing LDAPS grid-level data...")
ldaps_grid = prepare_weather_grid(ldaps_all, "ldaps")
print("Preparing GFS grid-level data...")
gfs_grid = prepare_weather_grid(gfs_all, "gfs")

ldaps_locations = ldaps_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
gfs_locations = gfs_grid[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)
display(ldaps_locations)
display(gfs_locations)

calendar_features = add_calendar_features(
    pd.concat([
        ldaps_grid[["forecast_kst_dtm"]],
        gfs_grid[["forecast_kst_dtm"]],
    ], ignore_index=True).drop_duplicates().sort_values("forecast_kst_dtm").reset_index(drop=True)
)

group_feature_sets = {}
feature_inventory_rows = []

print("LDAPS aggregation combination sizes:", LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES)

for target in TARGET_COLS:
    print(f"\nBuilding group-specific features for {target}")
    plan = LDAPS_GROUP_SPATIAL_PLAN[target]
    raw_ids = plan["raw_grid_ids"]
    subset_names = list(plan["subsets"].keys())

    ldaps_group_grid = add_target_wind_physics_features(ldaps_grid, LDAPS_UV_PAIRS, target)
    gfs_group_grid = add_target_wind_physics_features(gfs_grid, GFS_UV_PAIRS, target)
    ldaps_group_grid = add_group_speed_regime_flags(ldaps_group_grid, target, LDAPS_SPEED_COLS)
    gfs_group_grid = add_group_speed_regime_flags(gfs_group_grid, target, GFS_SPEED_COLS)

    group_frames = [calendar_features]

    if INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS:
        group_frames.append(aggregate_all_grids(ldaps_group_grid, "ldaps"))
    if INCLUDE_GFS_ALL_GRID_AGGREGATIONS:
        group_frames.append(aggregate_all_grids(gfs_group_grid, "gfs"))

    group_frames.extend([
        raw_grid_wide(gfs_group_grid, "gfs", GFS_RAW_GRID_IDS),
        gfs_grid_surface_pressure_diff(gfs_group_grid, high_grid_id=3, low_grid_id=5),
        raw_grid_wide(ldaps_group_grid, "ldaps", raw_ids),
    ])

    for subset_name, grid_ids in plan["subsets"].items():
        print(target, subset_name, "basic aggs:", grid_ids)
        group_frames.append(aggregate_subset(ldaps_group_grid, "ldaps", subset_name, grid_ids))

    features = merge_feature_frames(group_frames)
    features = features.sort_values("forecast_kst_dtm").reset_index(drop=True)

    lag_base_cols = choose_lag_roll_cols(features, subset_names)
    print(target, "lag/rolling base cols:", len(lag_base_cols))
    features = add_lag_rolling_features(features, lag_base_cols)

    train_df = labels.rename(columns={"kst_dtm": "forecast_kst_dtm"}).merge(features, on="forecast_kst_dtm", how="left")
    test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(features, on="forecast_kst_dtm", how="left")

    if len(test_df) != len(sample_submission):
        numeric_cols = [
            c for c in test_df.columns
            if c not in ["forecast_id", "forecast_kst_dtm"] and pd.api.types.is_numeric_dtype(test_df[c])
        ]
        collapsed = test_df.groupby(["forecast_id", "forecast_kst_dtm"], as_index=False)[numeric_cols].mean(numeric_only=True)
        test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
            collapsed,
            on=["forecast_id", "forecast_kst_dtm"],
            how="left",
        )
    assert len(test_df) == len(sample_submission)

    non_feature_cols = {"forecast_id", "forecast_kst_dtm", *TARGET_COLS}
    datetime_cols = [c for c in train_df.columns if str(train_df[c].dtype).startswith("datetime")]
    non_feature_cols.update(datetime_cols)
    feature_cols = [
        c for c in train_df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], ignore_index=True)
    keep_cols = [c for c in feature_cols if combined[c].notna().sum() > 0 and combined[c].nunique(dropna=True) > 1]
    del combined
    gc.collect()

    group_feature_sets[target] = {
        "features": features,
        "train_df": train_df,
        "test_df": test_df,
        "feature_cols": keep_cols,
        "lag_base_cols": lag_base_cols,
    }

    pd.DataFrame({"feature": keep_cols}).to_csv(
        FEATURE_OUTPUT_DIR / f"{target}_feature_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame({"feature": lag_base_cols}).to_csv(
        FEATURE_OUTPUT_DIR / f"{target}_lag_rolling_base_columns.csv",
        index=False,
        encoding="utf-8-sig",
    )
    feature_inventory_rows.append({
        "target": target,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "feature_count": len(keep_cols),
        "lag_base_count": len(lag_base_cols),
        "feature_selection_topk_importance": globals().get("FEATURE_SELECTION_TOPK_IMPORTANCE", None),
        "feature_selection_topk_shap": globals().get("FEATURE_SELECTION_TOPK_SHAP", None),
        "ldaps_aggregation_combination_sizes": json.dumps(LDAPS_GROUP_AGGREGATION_COMBINATION_SIZES),
        "raw_ldaps_grid_ids": json.dumps(raw_ids),
        "subset_names": json.dumps(subset_names),
        "count_regime_flags": COUNT_REGIME_FLAGS_IN_AGGREGATIONS,
        "count_precip_flags": COUNT_PRECIP_FLAGS_IN_AGGREGATIONS,
        "include_ldaps_all_grid_aggregations": INCLUDE_LDAPS_ALL_GRID_AGGREGATIONS,
        "include_gfs_all_grid_aggregations": INCLUDE_GFS_ALL_GRID_AGGREGATIONS,
        "raw_gfs_grid_ids": json.dumps(GFS_RAW_GRID_IDS),
        "gfs_surface_pressure_diff": "grid_3_minus_grid_5",
        "ldaps_blh_diff_features": True,
        "gfs_pbl_wind_and_vrate_features": True,
        "weather_lags": json.dumps(WEATHER_LAGS),
        "weather_roll_windows": json.dumps(WEATHER_ROLL_WINDOWS),
    })
    print(target, "train/test/features:", train_df.shape, test_df.shape, len(keep_cols))

feature_inventory = pd.DataFrame(feature_inventory_rows)
feature_inventory.to_csv(FEATURE_INVENTORY_PATH, index=False, encoding="utf-8-sig")
display(feature_inventory)
print("Feature build elapsed_sec:", round(time.time() - started, 1))
print("Saved feature inventory:", FEATURE_INVENTORY_PATH)


## (5) Feature Selection Helpers

This baseline uses the feature matrices produced by the preprocessing reference.   
소요시간: 약 1시간 내외, 53.0GB RAM (Colab L4 GPU) 기준


### Metrics, Splits, and Utilities

The grid search selects the best model per group using:

`0.5 * (1 - NMAE_g) + 0.5 * FICR_g`

The final validation report also computes the official-style average across the three groups.

In [ ]:
def kling_gupta_efficiency(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    actual = np.asarray(y_true, dtype=float)
    forecast = np.asarray(y_pred, dtype=float)
    finite = np.isfinite(actual) & np.isfinite(forecast)
    actual = actual[finite]
    forecast = forecast[finite]

    if len(actual) < 2:
        return {"kge": np.nan, "kge_corr": np.nan, "kge_alpha": np.nan, "kge_beta": np.nan}

    actual_mean = float(np.mean(actual))
    forecast_mean = float(np.mean(forecast))
    actual_std = float(np.std(actual))
    forecast_std = float(np.std(forecast))

    if actual_std <= 0 or actual_mean == 0:
        return {"kge": np.nan, "kge_corr": np.nan, "kge_alpha": np.nan, "kge_beta": np.nan}

    if forecast_std <= 0:
        corr = 0.0
    else:
        corr = float(np.corrcoef(actual, forecast)[0, 1])
        if not np.isfinite(corr):
            corr = 0.0

    alpha = forecast_std / actual_std
    beta = forecast_mean / actual_mean
    kge = 1.0 - np.sqrt((corr - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)

    return {
        "kge": float(kge),
        "kge_corr": float(corr),
        "kge_alpha": float(alpha),
        "kge_beta": float(beta),
    }


def group_metric(y_true: np.ndarray, y_pred: np.ndarray, target: str) -> dict:
    cap = CAPACITY_KWH[target]
    actual = np.asarray(y_true, dtype=float)
    forecast = np.asarray(y_pred, dtype=float)
    finite = np.isfinite(actual) & np.isfinite(forecast)
    actual = actual[finite]
    forecast = forecast[finite]
    if len(actual) == 0:
        return {
            "group_nmae": np.nan,
            "nmae_eval": np.nan,
            "one_minus_nmae": np.nan,
            "ficr": np.nan,
            "selection_score": np.nan,
            "mae_all_kwh": np.nan,
            "mae_eval_kwh": np.nan,
            "eval_rows": 0,
            "kge": np.nan,
            "kge_corr": np.nan,
            "kge_alpha": np.nan,
            "kge_beta": np.nan,
        }

    mae_all = float(np.mean(np.abs(forecast - actual)))
    eval_mask = actual >= cap * 0.10
    if eval_mask.sum() == 0:
        return {
            "group_nmae": np.nan,
            "nmae_eval": np.nan,
            "one_minus_nmae": np.nan,
            "ficr": np.nan,
            "selection_score": np.nan,
            "mae_all_kwh": mae_all,
            "mae_eval_kwh": np.nan,
            "eval_rows": 0,
            "kge": np.nan,
            "kge_corr": np.nan,
            "kge_alpha": np.nan,
            "kge_beta": np.nan,
        }

    actual_eval = actual[eval_mask]
    forecast_eval = forecast[eval_mask]
    abs_error_eval = np.abs(forecast_eval - actual_eval)
    error_rate = abs_error_eval / cap
    group_nmae = float(np.mean(error_rate))
    mae_eval_kwh = float(np.mean(abs_error_eval))
    nmae_eval = float(mae_eval_kwh / cap)
    one_minus_nmae = float(1.0 - group_nmae)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    earned = np.sum(actual_eval * unit_price)
    max_settlement = np.sum(actual_eval * 4.0)
    ficr = float(earned / max_settlement) if max_settlement > 0 else np.nan
    kge_values = kling_gupta_efficiency(actual_eval, forecast_eval)

    return {
        "group_nmae": group_nmae,
        "nmae_eval": nmae_eval,
        "one_minus_nmae": one_minus_nmae,
        "ficr": ficr,
        "selection_score": float(0.5 * one_minus_nmae + 0.5 * ficr),
        "mae_all_kwh": mae_all,
        "mae_eval_kwh": mae_eval_kwh,
        "eval_rows": int(eval_mask.sum()),
        "ficr_6pct_ratio": float(np.mean(error_rate <= 0.06)),
        "ficr_8pct_ratio": float(np.mean((error_rate > 0.06) & (error_rate <= 0.08))),
        "ficr_fail_ratio": float(np.mean(error_rate > 0.08)),
        **kge_values,
    }

def official_like_metric_from_long(pred_long: pd.DataFrame) -> dict:
    group_rows = []
    for target in TARGET_COLS:
        sub = pred_long[pred_long["group"] == target]
        if sub.empty:
            continue
        row = {"group": target, **group_metric(sub["actual_kwh"].values, sub["final_pred_kwh"].values, target)}
        group_rows.append(row)
    group_df = pd.DataFrame(group_rows)
    if group_df.empty:
        return {"total_score": np.nan, "one_minus_nmae": np.nan, "ficr": np.nan, "group_metrics": group_df}
    one_minus_nmae = float(1.0 - group_df["group_nmae"].mean())
    ficr = float(group_df["ficr"].mean())
    total_score = float(0.5 * one_minus_nmae + 0.5 * ficr)
    return {
        "total_score": total_score,
        "one_minus_nmae": one_minus_nmae,
        "ficr": ficr,
        "group_metrics": group_df,
    }


def full_2024_masks(df: pd.DataFrame, target: str) -> tuple[pd.Series, pd.Series]:
    labeled = df[target].notna()
    train_mask = labeled & (df["forecast_kst_dtm"] <= pd.Timestamp("2024-01-01 00:00:00"))
    valid_mask = (
        labeled
        & (df["forecast_kst_dtm"] >= pd.Timestamp("2024-01-01 01:00:00"))
        & (df["forecast_kst_dtm"] <= pd.Timestamp("2025-01-01 00:00:00"))
    )
    return train_mask, valid_mask


def final_label_mask(df: pd.DataFrame, target: str) -> pd.Series:
    return df[target].notna()


def clean_matrix(X: pd.DataFrame, medians: pd.Series | None = None) -> tuple[pd.DataFrame, pd.Series]:
    out = X.copy().replace([np.inf, -np.inf], np.nan)
    if medians is None:
        medians = out.median(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out = out.fillna(medians).fillna(0.0)
    return out.astype(np.float32), medians


def make_power_sample_weight(y: pd.Series, target: str) -> np.ndarray:
    cap = CAPACITY_KWH[target]
    ratio = (y.astype(float) / cap).clip(lower=0.0, upper=1.5)
    eval_mask = ratio >= 0.10
    weight = np.full(len(y), MIN_NON_EVAL_WEIGHT, dtype=float)
    weight[eval_mask] = 1.0
    weight += HIGH_POWER_WEIGHT_STRENGTH * np.square(np.minimum(ratio, 1.0))
    return np.asarray(weight, dtype=float)


def get_regressor_sample_weight(y: pd.Series, target: str) -> np.ndarray | None:
    if not USE_POWER_SAMPLE_WEIGHT:
        return None

    # weighted_l1 already embeds the same base weighting inside the custom objective.
    # Returning None here prevents applying the same weight twice.
    if REGRESSOR_OBJECTIVE == "weighted_l1":
        return None

    return make_power_sample_weight(y, target)


def param_grid(grid: dict, max_combos: int | None = None) -> list[dict]:
    keys = list(grid.keys())
    combos = [dict(zip(keys, values)) for values in itertools.product(*(grid[k] for k in keys))]
    if max_combos is not None and len(combos) > max_combos:
        rng = np.random.default_rng(RANDOM_STATE)
        chosen = sorted(rng.choice(len(combos), size=max_combos, replace=False).tolist())
        combos = [combos[i] for i in chosen]
    return combos


def stable_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, ensure_ascii=False)


def params_hash(obj) -> str:
    return hashlib.md5(stable_json(obj).encode("utf-8")).hexdigest()[:12]


def append_rows_csv(path: Path, rows: list[dict] | dict) -> None:
    if isinstance(rows, dict):
        rows = [rows]
    if not rows:
        return
    frame = pd.DataFrame(rows)
    write_header = not path.exists()
    frame.to_csv(path, mode="a", index=False, header=write_header, encoding="utf-8-sig")


def lgbm_callbacks(use_early_stopping: bool = True):
    callbacks = [lgb.log_evaluation(period=0)]
    if use_early_stopping and LGBM_EARLY_STOPPING_ROUNDS:
        callbacks.append(lgb.early_stopping(LGBM_EARLY_STOPPING_ROUNDS, verbose=False))
    return callbacks

### Feature Selection by Feature Importance, SHAP

In [ ]:
def assemble_power_frame(group: str) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    fs = group_feature_sets[group]
    feature_cols = [c for c in fs["feature_cols"] if pd.api.types.is_numeric_dtype(fs["train_df"][c])]
    train_frame = fs["train_df"][["forecast_kst_dtm", *TARGET_COLS, *feature_cols]].copy()
    test_frame = fs["test_df"][["forecast_id", "forecast_kst_dtm", *feature_cols]].copy()
    return train_frame, test_frame, feature_cols


def select_power_features(df: pd.DataFrame, candidate_cols: list[str], target: str, train_mask: pd.Series) -> list[str]:
    usable = [c for c in candidate_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

    # The final modeling feature set is determined by:
    # 1) LightGBM feature importance top FEATURE_SELECTION_TOPK_IMPORTANCE
    # 2) SHAP top FEATURE_SELECTION_TOPK_SHAP
    preselected_map = globals().get("PRESELECTED_FEATURES_BY_GROUP", {})
    preselected = preselected_map.get(target)
    if preselected:
        selected = [c for c in preselected if c in usable]
        if selected:
            return selected

    # This fallback is used only before the feature-selection cell has been executed
    # so that feature inventory/audit cells can inspect the raw candidate feature space.
    return usable


feature_audit_rows = []
for group in TARGET_COLS:
    train_frame, test_frame, candidate_cols = assemble_power_frame(group)
    train_mask, valid_mask = full_2024_masks(train_frame, group)
    feature_audit_rows.append({
        "group": group,
        "candidate_features_before_selection": len(candidate_cols),
        "feature_importance_topk": min(FEATURE_SELECTION_TOPK_IMPORTANCE, len(candidate_cols)),
        "shap_final_topk": min(FEATURE_SELECTION_TOPK_SHAP, len(candidate_cols)),
        "train_rows": int(train_mask.sum()),
        "valid_rows": int(valid_mask.sum()),
        "test_rows": len(test_frame),
    })

feature_audit = pd.DataFrame(feature_audit_rows)
display(feature_audit)


## (6) Train Model

Model I: `Pr(generation < 100 kWh)`.

Model II: `Pr(generation >= CAPACITY_LABEL_RATIO * group capacity kWh)` (default - 0.98).

Model III: `continuous generation regressor` (loss function - weighted L1 or weighted L2 or custom loss).


### Three-Model AutoGluon Regime Pipeline

In [ ]:

class ConstantBinaryModel:
    def __init__(self, probability: float):
        self.probability = float(probability)

    def predict_proba(self, X):
        p = np.full(len(X), self.probability, dtype=float)
        return np.column_stack([1.0 - p, p])

    def predict(self, X):
        return np.full(len(X), int(self.probability >= 0.5), dtype=int)


def class_balance_weight(y: pd.Series | np.ndarray) -> np.ndarray:
    arr = np.asarray(y, dtype=int)
    pos = max(int(arr.sum()), 1)
    neg = max(int((1 - arr).sum()), 1)
    pos_w = neg / pos
    weight = np.where(arr == 1, pos_w, 1.0).astype(float)
    return normalize_sample_weight(weight)


def normalize_sample_weight(weight: np.ndarray) -> np.ndarray:
    weight = np.asarray(weight, dtype=float)
    weight = np.where(np.isfinite(weight) & (weight > 0), weight, 1.0)
    total = float(weight.sum())
    if total <= 0:
        return np.ones(len(weight), dtype=float)
    return weight * (len(weight) / total)


def component_target_values(component: str, y: pd.Series, group: str) -> pd.Series:
    cap = CAPACITY_KWH[group]
    y = y.astype(float)
    if component == "model1_zero":
        return (y < ZERO_LABEL_KWH).astype(int)
    if component == "model2_capacity":
        return (y >= cap * CAPACITY_LABEL_RATIO).astype(int)
    if component == "model3_regressor":
        return y
    raise ValueError(f"Unknown component: {component}")


def component_problem_type(component: str) -> str:
    return "regression" if component == "model3_regressor" else "binary"


def component_eval_metric(component: str) -> str:
    return "mean_absolute_error" if component == "model3_regressor" else "log_loss"


def component_sample_weight(component: str, y_raw: pd.Series, y_component: pd.Series, group: str) -> np.ndarray | None:
    if not USE_POWER_SAMPLE_WEIGHT:
        return None
    if component == "model3_regressor":
        return normalize_sample_weight(make_power_sample_weight(y_raw, group))
    return class_balance_weight(y_component)

def fit_lightgbm_zero_component(
    group: str,
    X_train: pd.DataFrame,
    y_train_raw: pd.Series,
    X_valid: pd.DataFrame | None = None,
    y_valid_raw: pd.Series | None = None,
):
    y_train_zero = component_target_values("model1_zero", y_train_raw, group).astype(int)
    unique = np.unique(np.asarray(y_train_zero, dtype=int))
    if len(unique) < 2:
        print(f"[WARN] {group}/model1_zero: only one class in train. Using constant model.")
        return ConstantBinaryModel(float(unique[0]) if len(unique) else 0.0)

    model = LGBMClassifier(
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        force_col_wise=True,
        verbosity=-1,
        **ZERO_CLASSIFIER_PARAMS,
    )

    eval_set = None
    eval_metric = None
    if X_valid is not None and y_valid_raw is not None:
        y_valid_zero = component_target_values("model1_zero", y_valid_raw, group).astype(int)
        if len(np.unique(np.asarray(y_valid_zero, dtype=int))) >= 2:
            eval_set = [(X_valid, y_valid_zero)]
            eval_metric = "binary_logloss"

    model.fit(
        X_train,
        y_train_zero,
        sample_weight=class_balance_weight(y_train_zero),
        eval_set=eval_set,
        eval_metric=eval_metric,
        callbacks=[lgb.log_evaluation(0)],
    )
    return model



def autogluon_component_path(group: str, component: str, phase: str) -> Path:
    return AUTOGLUON_MODEL_DIR / phase / group / component


def make_ag_frame(
    X: pd.DataFrame,
    y_component: pd.Series | np.ndarray | None = None,
    sample_weight: np.ndarray | None = None,
) -> pd.DataFrame:
    frame = X.reset_index(drop=True).copy()
    if y_component is not None:
        frame[AG_LABEL_COL] = np.asarray(y_component)
    if sample_weight is not None:
        frame[AG_SAMPLE_WEIGHT_COL] = normalize_sample_weight(sample_weight)
    return frame


def fit_autogluon_component(
    group: str,
    component: str,
    X_train: pd.DataFrame,
    y_train_raw: pd.Series,
    X_valid: pd.DataFrame | None,
    y_valid_raw: pd.Series | None,
    phase: str,
):
    y_train_component = component_target_values(component, y_train_raw, group)
    problem_type = component_problem_type(component)
    eval_metric = component_eval_metric(component)

    if problem_type == "binary":
        unique = np.unique(np.asarray(y_train_component, dtype=int))
        if len(unique) < 2:
            print(f"[WARN] {group}/{component}: only one class in train. Using constant model.")
            return ConstantBinaryModel(float(unique[0]) if len(unique) else 0.0), None

    train_weight = component_sample_weight(component, y_train_raw, y_train_component, group)
    train_ag = make_ag_frame(X_train, y_train_component, train_weight)

    tuning_ag = None
    if X_valid is not None and y_valid_raw is not None:
        y_valid_component = component_target_values(component, y_valid_raw, group)
        valid_weight = component_sample_weight(component, y_valid_raw, y_valid_component, group)
        tuning_ag = make_ag_frame(X_valid, y_valid_component, valid_weight)

    path = autogluon_component_path(group, component, phase)
    if path.exists() and not FORCE_RERUN_AUTOGLUON:
        try:
            predictor = TabularPredictor.load(str(path))
            print(f"[AUTOGLUON CACHE HIT] {group}/{component}/{phase}: {path}")
            return predictor, tuning_ag
        except Exception as e:
            print(f"[WARN] Failed to load existing AutoGluon predictor at {path}: {e}")

    if path.exists() and FORCE_RERUN_AUTOGLUON:
        shutil.rmtree(path)

    predictor = TabularPredictor(
        label=AG_LABEL_COL,
        problem_type=problem_type,
        eval_metric=eval_metric,
        path=str(path),
        sample_weight=AG_SAMPLE_WEIGHT_COL if train_weight is not None else None,
        weight_evaluation=AUTOGLUON_WEIGHT_EVALUATION,
        verbosity=AUTOGLUON_VERBOSITY,
    )
    predictor.fit(
        train_data=train_ag,
        tuning_data=tuning_ag,
        presets=AUTOGLUON_PRESETS,
        hyperparameters=AUTOGLUON_HYPERPARAMETERS,
        hyperparameter_tune_kwargs=AUTOGLUON_HYPERPARAMETER_TUNE_KWARGS,
        fit_weighted_ensemble=True,
        num_bag_folds=AUTOGLUON_NUM_BAG_FOLDS,
        num_stack_levels=AUTOGLUON_NUM_STACK_LEVELS,
        num_gpus=AUTOGLUON_NUM_GPUS,
        time_limit=AUTOGLUON_TIME_LIMIT_PER_COMPONENT,
    )
    return predictor, tuning_ag


def positive_proba_from_predictor(model, X: pd.DataFrame) -> np.ndarray:
    if isinstance(model, ConstantBinaryModel):
        return model.predict_proba(X)[:, 1]
    proba = model.predict_proba(X)
    if isinstance(proba, pd.DataFrame):
        for col in [1, True, "1", "True"]:
            if col in proba.columns:
                return proba[col].to_numpy(dtype=float)
        return proba.iloc[:, -1].to_numpy(dtype=float)
    arr = np.asarray(proba, dtype=float)
    if arr.ndim == 2:
        return arr[:, -1]
    return arr


def predict_regime_pipeline(artifact: dict, X: pd.DataFrame, group: str) -> dict:
    cap = CAPACITY_KWH[group]
    zero_prob = positive_proba_from_predictor(artifact["zero_model"], X)
    capacity_prob = positive_proba_from_predictor(artifact["capacity_model"], X)
    reg_pred = np.asarray(artifact["reg_model"].predict(X), dtype=float)
    reg_pred = np.clip(reg_pred, 0.0, cap)
    final_pred, zero_flag, capacity_flag = apply_regime_rule(zero_prob, capacity_prob, reg_pred, cap)
    return {
        "zero_prob": zero_prob,
        "capacity_prob": capacity_prob,
        "reg_pred": reg_pred,
        "final_pred": final_pred,
        "zero_flag": zero_flag,
        "capacity_flag": capacity_flag,
    }


def apply_regime_rule(zero_prob, capacity_prob, reg_pred, capacity):
    zero_flag = np.asarray(zero_prob) >= ZERO_THRESHOLD
    capacity_flag = np.asarray(capacity_prob) >= CAPACITY_THRESHOLD
    pred = np.asarray(reg_pred, dtype=float).copy()
    pred[zero_flag & ~capacity_flag] = 0.0
    pred[capacity_flag & ~zero_flag] = capacity
    pred = np.clip(pred, 0.0, capacity)
    return pred, zero_flag, capacity_flag


def safe_autogluon_leaderboard(model, data: pd.DataFrame | None, group: str, component: str, phase: str) -> pd.DataFrame:
    if isinstance(model, ConstantBinaryModel):
        return pd.DataFrame({
            "group": [group],
            "component": [component],
            "phase": [phase],
            "model": ["ConstantBinaryModel"],
            "score_val": [np.nan],
        })
    if not hasattr(model, "leaderboard"):
        return pd.DataFrame({
            "group": [group],
            "component": [component],
            "phase": [phase],
            "model": [type(model).__name__],
            "score_val": [np.nan],
        })
    try:
        lb = model.leaderboard(data, silent=True)
    except Exception:
        try:
            lb = model.leaderboard(silent=True)
        except Exception as e:
            print(f"[WARN] leaderboard failed for {group}/{component}/{phase}: {e}")
            return pd.DataFrame()
    lb = lb.copy()
    lb.insert(0, "group", group)
    lb.insert(1, "component", component)
    lb.insert(2, "phase", phase)
    return lb


### Feature Selection Helpers

In [ ]:

# =============================
# Train-only feature selection
# =============================

FEATURE_SELECTION_COMPONENTS = [
    ("model1_zero", "classifier_zero"),
    ("model2_capacity", "classifier_capacity"),
    ("model3_regressor", "regressor"),
]


def cap_selector_estimators(params: dict) -> dict:
    params = dict(params)
    if "n_estimators" in params:
        params["n_estimators"] = int(min(params["n_estimators"], FEATURE_SELECTION_MAX_ESTIMATORS))
    return params


def fit_feature_selector_model(component: str, X: pd.DataFrame, y_component: pd.Series, group: str):
    callbacks = [lgb.log_evaluation(0)]

    if component in {"model1_zero", "model2_capacity"}:
        y_arr = np.asarray(y_component, dtype=int)
        unique = np.unique(y_arr)
        if len(unique) < 2:
            return ConstantBinaryModel(float(unique[0]) if len(unique) else 0.0)
        params = cap_selector_estimators(FEATURE_SELECTION_CLASSIFIER_PARAMS)
        model = LGBMClassifier(
            objective="binary",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            force_col_wise=True,
            verbosity=-1,
            **params,
        )
        model.fit(
            X,
            y_arr,
            sample_weight=class_balance_weight(y_arr),
            eval_metric="binary_logloss",
            callbacks=callbacks,
        )
        return model

    params = cap_selector_estimators(FEATURE_SELECTION_REGRESSOR_PARAMS)
    model = LGBMRegressor(
        objective="regression_l1",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        force_col_wise=True,
        verbosity=-1,
        **params,
    )
    sample_weight = normalize_sample_weight(make_power_sample_weight(y_component.astype(float), group)) if USE_POWER_SAMPLE_WEIGHT else None
    model.fit(
        X,
        y_component.astype(float),
        sample_weight=sample_weight,
        eval_metric="l1",
        callbacks=callbacks,
    )
    return model


def model_importance_frame(model, features: list[str], group: str, component: str) -> pd.DataFrame:
    if isinstance(model, ConstantBinaryModel) or not hasattr(model, "feature_importances_"):
        importance = np.ones(len(features), dtype=float)
    else:
        importance = np.asarray(model.feature_importances_, dtype=float)
        if len(importance) != len(features):
            importance = np.resize(importance, len(features))

    out = pd.DataFrame({
        "group": group,
        "component": component,
        "feature": features,
        "importance": importance,
    })
    out["importance"] = out["importance"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out = out.sort_values(["importance", "feature"], ascending=[False, True]).reset_index(drop=True)
    out["importance_rank"] = np.arange(1, len(out) + 1)
    return out


def shap_importance_frame(model, X: pd.DataFrame, group: str, component: str) -> pd.DataFrame:
    if isinstance(model, ConstantBinaryModel):
        return pd.DataFrame({
            "group": group,
            "component": component,
            "feature": X.columns.tolist(),
            "mean_abs_shap": np.zeros(X.shape[1], dtype=float),
            "sample_rows": len(X),
        })

    if not SHAP_AVAILABLE:
        return pd.DataFrame()

    if len(X) > FEATURE_SELECTION_SHAP_SAMPLE_SIZE:
        X_sample = X.sample(FEATURE_SELECTION_SHAP_SAMPLE_SIZE, random_state=RANDOM_STATE)
    else:
        X_sample = X.copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mean_abs = np.abs(shap_values).mean(axis=0)
    out = pd.DataFrame({
        "group": group,
        "component": component,
        "feature": X_sample.columns.tolist(),
        "mean_abs_shap": mean_abs,
        "sample_rows": len(X_sample),
    })
    out["mean_abs_shap"] = out["mean_abs_shap"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out = out.sort_values(["mean_abs_shap", "feature"], ascending=[False, True]).reset_index(drop=True)
    out["shap_rank"] = np.arange(1, len(out) + 1)
    return out


def combine_component_shap_rankings(shap_frames: list[pd.DataFrame], group: str) -> pd.DataFrame:
    if not shap_frames:
        return pd.DataFrame(columns=["feature", "combined_rank_score", "component_count"])

    frames = []
    for frame in shap_frames:
        tmp = frame.copy()
        if "shap_rank" not in tmp.columns:
            tmp = tmp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
            tmp["shap_rank"] = np.arange(1, len(tmp) + 1)
        denom = max(len(tmp) - 1, 1)
        tmp["rank_score"] = 1.0 - ((tmp["shap_rank"] - 1) / denom)
        frames.append(tmp[["group", "component", "feature", "mean_abs_shap", "shap_rank", "rank_score"]])

    long = pd.concat(frames, ignore_index=True)
    combined = (
        long.groupby("feature", as_index=False)
        .agg(
            combined_rank_score=("rank_score", "mean"),
            max_rank_score=("rank_score", "max"),
            mean_abs_shap_mean=("mean_abs_shap", "mean"),
            mean_abs_shap_max=("mean_abs_shap", "max"),
            component_count=("component", "nunique"),
        )
        .sort_values(
            ["combined_rank_score", "max_rank_score", "component_count", "feature"],
            ascending=[False, False, False, True],
        )
        .reset_index(drop=True)
    )
    combined.insert(0, "group", group)
    combined["combined_rank"] = np.arange(1, len(combined) + 1)
    return combined


def feature_selection_cache_paths(group: str) -> tuple[Path, Path]:
    combined_path = FEATURE_SELECTION_DIR / f"{group}_combined_shap_top{FEATURE_SELECTION_TOPK_SHAP}.csv"
    meta_path = FEATURE_SELECTION_DIR / f"{group}_feature_selection_cache_meta.json"
    return combined_path, meta_path


def feature_selection_cache_signature(group: str, usable_features: list[str], train_mask: pd.Series) -> str:
    signature_payload = {
        "group": group,
        "validation_option": VALIDATION_OPTION,
        "feature_selection_topk_importance": FEATURE_SELECTION_TOPK_IMPORTANCE,
        "feature_selection_topk_shap": FEATURE_SELECTION_TOPK_SHAP,
        "candidate_feature_count": len(usable_features),
        "candidate_features": usable_features,
        "train_rows": int(train_mask.sum()),
        "model_components": [component for component, _ in FEATURE_SELECTION_COMPONENTS],
    }
    return params_hash(signature_payload)


def save_feature_selection_cache_metadata(
    group: str,
    usable_features: list[str],
    train_mask: pd.Series,
    selected_features: list[str],
    combined_path: Path,
) -> None:
    _, meta_path = feature_selection_cache_paths(group)
    metadata = {
        "group": group,
        "cache_signature": feature_selection_cache_signature(group, usable_features, train_mask),
        "validation_option": VALIDATION_OPTION,
        "feature_selection_topk_importance": FEATURE_SELECTION_TOPK_IMPORTANCE,
        "feature_selection_topk_shap": FEATURE_SELECTION_TOPK_SHAP,
        "candidate_feature_count": len(usable_features),
        "train_rows": int(train_mask.sum()),
        "selected_feature_count": len(selected_features),
        "combined_csv": str(combined_path),
    }
    meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")


def load_feature_selection_cache_for_group(group: str) -> list[str] | None:
    df, _, candidate_cols = assemble_power_frame(group)
    train_mask, _ = full_2024_masks(df, group)
    usable = [c for c in candidate_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    expected_signature = feature_selection_cache_signature(group, usable, train_mask)
    combined_path, meta_path = feature_selection_cache_paths(group)

    if not combined_path.exists():
        print(f"[CACHE MISS] {group}: feature-selection cache CSV is missing.")
        return None

    if not meta_path.exists():
        if not ALLOW_LEGACY_FEATURE_SELECTION_CACHE:
            print(f"[CACHE MISS] {group}: cache metadata is missing.")
            return None
        print(f"[CACHE LEGACY] {group}: metadata is missing, loading CSV after feature-name validation.")
        cached = pd.read_csv(combined_path)
        if "feature" not in cached.columns:
            print(f"[CACHE MISS] {group}: cached CSV has no feature column.")
            return None
        usable_set = set(usable)
        selected = [
            feature
            for feature in cached["feature"].dropna().astype(str).head(FEATURE_SELECTION_TOPK_SHAP).tolist()
            if feature in usable_set
        ]
        if len(selected) == 0:
            print(f"[CACHE MISS] {group}: no cached feature is usable in the current dataframe.")
            return None
        print(f"[CACHE HIT] {group}: loaded {len(selected):,} legacy selected features from {combined_path}")
        return selected

    try:
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[CACHE MISS] {group}: failed to read cache metadata: {e}")
        return None

    if metadata.get("cache_signature") != expected_signature:
        print(f"[CACHE MISS] {group}: preprocessing/top-k signature changed.")
        print("  cached:", metadata.get("cache_signature"))
        print("  current:", expected_signature)
        return None

    cached = pd.read_csv(combined_path)
    if "feature" not in cached.columns:
        print(f"[CACHE MISS] {group}: cached CSV has no feature column.")
        return None

    usable_set = set(usable)
    selected = [
        feature
        for feature in cached["feature"].dropna().astype(str).head(FEATURE_SELECTION_TOPK_SHAP).tolist()
        if feature in usable_set
    ]

    if len(selected) == 0:
        print(f"[CACHE MISS] {group}: no cached feature is usable in the current dataframe.")
        return None

    print(f"[CACHE HIT] {group}: loaded {len(selected):,} selected features from {combined_path}")
    return selected


def load_preselected_features_from_cache() -> dict[str, list[str]] | None:
    selected_by_group = {}

    for group in TARGET_COLS:
        selected = load_feature_selection_cache_for_group(group)
        if selected is None:
            return None
        selected_by_group[group] = selected

    summary = pd.DataFrame({
        "group": list(selected_by_group.keys()),
        "selected_features": [len(v) for v in selected_by_group.values()],
        "loaded_from_cache": True,
    })
    display(summary)
    return selected_by_group


def run_shap_feature_selection_for_group(group: str) -> list[str]:
    df, _, candidate_cols = assemble_power_frame(group)
    train_mask, _ = full_2024_masks(df, group)
    usable = [c for c in candidate_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    X_all, _ = clean_matrix(df.loc[train_mask, usable])
    y_raw = df.loc[train_mask, group].astype(float)

    component_shap_frames = []
    component_top_rows = []

    for component, _ in FEATURE_SELECTION_COMPONENTS:
        print(f"Feature selection: {group} / {component} - fitting importance model")
        y_component = component_target_values(component, y_raw, group)
        model_all = fit_feature_selector_model(component, X_all, y_component, group)
        fi = model_importance_frame(model_all, usable, group, component)
        fi_path = FEATURE_SELECTION_DIR / f"{group}_{component}_importance_top{FEATURE_SELECTION_TOPK_IMPORTANCE}.csv"
        fi.head(FEATURE_SELECTION_TOPK_IMPORTANCE).to_csv(fi_path, index=False, encoding="utf-8-sig")

        top_importance_features = fi.head(FEATURE_SELECTION_TOPK_IMPORTANCE)["feature"].tolist()
        X_top = X_all[top_importance_features]

        print(f"Feature selection: {group} / {component} - fitting SHAP model on {len(top_importance_features):,} features")
        model_top = fit_feature_selector_model(component, X_top, y_component, group)
        shap_frame = shap_importance_frame(model_top, X_top, group, component)

        if shap_frame.empty:
            print(f"[WARN] SHAP unavailable for {group} / {component}; falling back to feature importance ranking.")
            shap_frame = fi.head(FEATURE_SELECTION_TOPK_IMPORTANCE).rename(
                columns={"importance": "mean_abs_shap", "importance_rank": "shap_rank"}
            )[["group", "component", "feature", "mean_abs_shap", "shap_rank"]].copy()
            shap_frame["sample_rows"] = len(X_top)

        shap_path = FEATURE_SELECTION_DIR / f"{group}_{component}_shap_top{FEATURE_SELECTION_TOPK_SHAP}.csv"
        shap_frame.head(FEATURE_SELECTION_TOPK_SHAP).to_csv(shap_path, index=False, encoding="utf-8-sig")
        component_shap_frames.append(shap_frame)
        component_top_rows.append({
            "group": group,
            "component": component,
            "candidate_features": len(usable),
            "importance_candidates": len(top_importance_features),
            "shap_ranked_features": len(shap_frame),
            "saved_top_shap_features": min(FEATURE_SELECTION_TOPK_SHAP, len(shap_frame)),
            "importance_csv": str(fi_path),
            "shap_csv": str(shap_path),
        })

        del model_all, model_top, X_top
        gc.collect()

    combined = combine_component_shap_rankings(component_shap_frames, group)
    selected = combined.head(FEATURE_SELECTION_TOPK_SHAP)["feature"].tolist()
    combined_path = FEATURE_SELECTION_DIR / f"{group}_combined_shap_top{FEATURE_SELECTION_TOPK_SHAP}.csv"
    combined.head(FEATURE_SELECTION_TOPK_SHAP).to_csv(combined_path, index=False, encoding="utf-8-sig")
    save_feature_selection_cache_metadata(group, usable, train_mask, selected, combined_path)

    summary = pd.DataFrame(component_top_rows)
    summary["combined_csv"] = str(combined_path)
    summary["final_selected_features"] = len(selected)
    summary.to_csv(FEATURE_SELECTION_DIR / f"{group}_feature_selection_summary.csv", index=False, encoding="utf-8-sig")

    del X_all
    gc.collect()
    return selected


def run_shap_feature_selection() -> dict[str, list[str]]:
    selected_by_group = {}
    summary_rows = []

    for group in TARGET_COLS:
        selected = run_shap_feature_selection_for_group(group)
        selected_by_group[group] = selected
        summary_rows.append({
            "group": group,
            "selected_features": len(selected),
            "top_feature_1": selected[0] if selected else None,
            "top_feature_2": selected[1] if len(selected) > 1 else None,
            "top_feature_3": selected[2] if len(selected) > 2 else None,
        })
        print(f"Feature selection complete: {group}, selected={len(selected):,}")

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(FEATURE_SELECTION_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    display(summary)
    return selected_by_group


### Run AutoGluon Training

In [ ]:

if RUN_SHAP_FEATURE_SELECTION:
    PRESELECTED_FEATURES_BY_GROUP = None

    if REUSE_FEATURE_SELECTION_CACHE and not FORCE_RERUN_FEATURE_SELECTION:
        PRESELECTED_FEATURES_BY_GROUP = load_preselected_features_from_cache()

    if PRESELECTED_FEATURES_BY_GROUP is None:
        print("Running feature selection from scratch.")
        PRESELECTED_FEATURES_BY_GROUP = run_shap_feature_selection()
else:
    PRESELECTED_FEATURES_BY_GROUP = {}
    print("SHAP feature selection skipped. All usable candidate features will be used.")


def fit_regime_pipeline(group: str, use_validation_fit: bool, phase: str):
    df, test_df, candidate_cols = assemble_power_frame(group)
    if use_validation_fit:
        train_mask, valid_mask = full_2024_masks(df, group)
    else:
        train_mask = final_label_mask(df, group)
        valid_mask = pd.Series(False, index=df.index)

    selected = select_power_features(df, candidate_cols, group, train_mask)
    X_train_raw = df.loc[train_mask, selected]
    y_train = df.loc[train_mask, group].astype(float)
    X_train, medians = clean_matrix(X_train_raw)

    if use_validation_fit:
        X_valid_raw = df.loc[valid_mask, selected]
        y_valid = df.loc[valid_mask, group].astype(float)
        X_valid, _ = clean_matrix(X_valid_raw, medians)
    else:
        X_valid = None
        y_valid = None

    zero_model = fit_lightgbm_zero_component(group, X_train, y_train, X_valid, y_valid)
    zero_tuning = None
    capacity_model, capacity_tuning = fit_autogluon_component(
        group, "model2_capacity", X_train, y_train, X_valid, y_valid, phase
    )
    reg_model, reg_tuning = fit_autogluon_component(
        group, "model3_regressor", X_train, y_train, X_valid, y_valid, phase
    )

    return {
        "df": df,
        "test_df": test_df,
        "selected": selected,
        "medians": medians,
        "zero_model": zero_model,
        "capacity_model": capacity_model,
        "reg_model": reg_model,
        "train_mask": train_mask,
        "valid_mask": valid_mask,
        "phase": phase,
        "tuning_frames": {
            "model1_zero": zero_tuning,
            "model2_capacity": capacity_tuning,
            "model3_regressor": reg_tuning,
        },
    }


validation_artifacts = {}
validation_prediction_frames = []
validation_metric_rows = []
wrong_case_frames = []
leaderboard_frames = []

if RUN_AUTOGLUON_TRAIN:
    for group in TARGET_COLS:
        print(f"\n===== AutoGluon validation fit: {group} =====")
        artifact = fit_regime_pipeline(group, use_validation_fit=True, phase="validation")
        df = artifact["df"]
        valid_mask = artifact["valid_mask"]
        selected = artifact["selected"]
        cap = CAPACITY_KWH[group]
        X_valid, _ = clean_matrix(df.loc[valid_mask, selected], artifact["medians"])
        y_valid = df.loc[valid_mask, group].astype(float)
        pred = predict_regime_pipeline(artifact, X_valid, group)
        metrics = group_metric(y_valid.values, pred["final_pred"], group)
        validation_metric_rows.append({"group": group, "n_features": len(selected), **metrics})

        true_zero = y_valid.values < ZERO_LABEL_KWH
        true_capacity = y_valid.values >= cap * CAPACITY_LABEL_RATIO
        pred_frame = pd.DataFrame({
            "forecast_kst_dtm": df.loc[valid_mask, "forecast_kst_dtm"].values,
            "group": group,
            "actual_kwh": y_valid.values,
            "zero_prob": pred["zero_prob"],
            "capacity_prob": pred["capacity_prob"],
            "zero_flag": pred["zero_flag"],
            "capacity_flag": pred["capacity_flag"],
            "true_zero": true_zero,
            "true_capacity": true_capacity,
            "model1_zero_pred_kwh": np.zeros(len(y_valid), dtype=float),
            "model2_capacity_pred_kwh": np.full(len(y_valid), cap, dtype=float),
            "model3_reg_pred_kwh": pred["reg_pred"],
            "final_pred_kwh": pred["final_pred"],
        })
        pred_frame["model1_zero_residual_kwh"] = pred_frame["model1_zero_pred_kwh"] - pred_frame["actual_kwh"]
        pred_frame["model2_capacity_residual_kwh"] = pred_frame["model2_capacity_pred_kwh"] - pred_frame["actual_kwh"]
        pred_frame["model3_reg_residual_kwh"] = pred_frame["model3_reg_pred_kwh"] - pred_frame["actual_kwh"]
        pred_frame["final_residual_kwh"] = pred_frame["final_pred_kwh"] - pred_frame["actual_kwh"]
        pred_frame["residual_kwh"] = pred_frame["final_residual_kwh"]
        pred_frame["residual_capacity_ratio"] = pred_frame["final_residual_kwh"] / cap
        validation_prediction_frames.append(pred_frame)

        case_feature_cols = selected[:CASE_EXPORT_MAX_FEATURES]
        case_base = df.loc[valid_mask, ["forecast_kst_dtm", group, *case_feature_cols]].copy()
        case_base = case_base.rename(columns={group: "actual_kwh"}).reset_index(drop=True)
        case_base["group"] = group
        case_base["zero_prob"] = pred["zero_prob"]
        case_base["capacity_prob"] = pred["capacity_prob"]
        case_base["zero_flag"] = pred["zero_flag"]
        case_base["capacity_flag"] = pred["capacity_flag"]
        case_base["true_zero"] = true_zero
        case_base["true_capacity"] = true_capacity
        case_base["model3_reg_pred_kwh"] = pred["reg_pred"]
        case_base["final_pred_kwh"] = pred["final_pred"]

        zero_wrong = case_base[case_base["zero_flag"].astype(bool) != case_base["true_zero"].astype(bool)].copy()
        zero_wrong["wrong_model"] = "model1_zero"
        capacity_wrong = case_base[case_base["capacity_flag"].astype(bool) != case_base["true_capacity"].astype(bool)].copy()
        capacity_wrong["wrong_model"] = "model2_capacity"
        wrong_case_frames.extend([zero_wrong, capacity_wrong])

        for component, model_key in [
            ("model1_zero", "zero_model"),
            ("model2_capacity", "capacity_model"),
            ("model3_regressor", "reg_model"),
        ]:
            leaderboard_frames.append(
                safe_autogluon_leaderboard(
                    artifact[model_key],
                    artifact["tuning_frames"].get(component),
                    group,
                    component,
                    "validation",
                )
            )

        validation_artifacts[group] = artifact
        print(group, metrics)

    validation_predictions = pd.concat(validation_prediction_frames, ignore_index=True)
    validation_metrics = pd.DataFrame(validation_metric_rows)
    wrong_regime_cases = pd.concat(wrong_case_frames, ignore_index=True) if wrong_case_frames else pd.DataFrame()
    autogluon_leaderboard = pd.concat([f for f in leaderboard_frames if f is not None and not f.empty], ignore_index=True) if leaderboard_frames else pd.DataFrame()

    official_validation = official_like_metric_from_long(validation_predictions)
    official_group_metrics = official_validation["group_metrics"]

    validation_predictions.to_csv(VALID_PRED_PATH, index=False, encoding="utf-8-sig")
    validation_metrics.to_csv(VALID_METRIC_PATH, index=False, encoding="utf-8-sig")
    wrong_regime_cases.to_csv(WRONG_CASE_PATH, index=False, encoding="utf-8-sig")
    official_group_metrics.to_csv(CHECKPOINT_DIR / "official_like_group_metrics.csv", index=False, encoding="utf-8-sig")
    validation_metrics.to_csv(AUTOGLUON_VALID_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    autogluon_leaderboard.to_csv(AUTOGLUON_LEADERBOARD_PATH, index=False, encoding="utf-8-sig")

    best_power_rows = validation_metrics.copy()
    best_power_rows.to_csv(BEST_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    display(validation_metrics)
    display(official_group_metrics)
    if not autogluon_leaderboard.empty:
        display(autogluon_leaderboard.groupby(["group", "component"]).head(8))
    if len(wrong_regime_cases):
        display(wrong_regime_cases.groupby(["group", "wrong_model"]).size().reset_index(name="n_wrong"))

    print("Official-like validation total_score:", official_validation["total_score"])
    print("Official-like 1-NMAE:", official_validation["one_minus_nmae"])
    print("Official-like FICR:", official_validation["ficr"])
    print("Saved validation predictions:", VALID_PRED_PATH)
    print("Saved AutoGluon validation summary:", AUTOGLUON_VALID_SUMMARY_PATH)
    print("Saved AutoGluon leaderboard:", AUTOGLUON_LEADERBOARD_PATH)
else:
    print("RUN_AUTOGLUON_TRAIN is False. AutoGluon validation fit skipped.")


### Validation Model Summary

In [ ]:

if "validation_metrics" in globals():
    display(validation_metrics)
if "official_group_metrics" in globals():
    display(official_group_metrics)
if "autogluon_leaderboard" in globals() and not autogluon_leaderboard.empty:
    display(autogluon_leaderboard.groupby(["group", "component"]).head(5))


## (7) Validation Diagnostics


### Residual Distributions

Note: `Residual = pred - actual`  
Residuals are shown for the final decision and each model component.

In [ ]:
def build_component_residual_long(validation_predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    component_specs = [
        ("model1_zero", "model1_zero_residual_kwh", "zero_flag"),
        ("model2_capacity", "model2_capacity_residual_kwh", "capacity_flag"),
        ("model3_regressor", "model3_reg_residual_kwh", None),
        ("final_decision", "final_residual_kwh", None),
    ]
    for component, residual_col, fire_col in component_specs:
        cols = ["forecast_kst_dtm", "group", "actual_kwh", residual_col]
        if fire_col is not None:
            cols.append(fire_col)
        sub = validation_predictions[cols].copy()
        if fire_col is not None:
            sub = sub[sub[fire_col].astype(bool)].copy()
        sub = sub.rename(columns={residual_col: "residual_kwh"})
        sub["component"] = component
        rows.append(sub[["forecast_kst_dtm", "group", "actual_kwh", "residual_kwh", "component"]])
    return pd.concat(rows, ignore_index=True)


residual_long = validation_predictions.copy()
residual_long["forecast_kst_dtm"] = pd.to_datetime(residual_long["forecast_kst_dtm"])
residual_long["capacity_kwh"] = residual_long["group"].map(CAPACITY_KWH)
residual_long["abs_residual_kwh"] = residual_long["final_residual_kwh"].abs()
residual_long["abs_error_rate"] = residual_long["abs_residual_kwh"] / residual_long["capacity_kwh"]
residual_long["actual_capacity_ratio"] = residual_long["actual_kwh"] / residual_long["capacity_kwh"]
residual_long["eval_mask"] = residual_long["actual_kwh"] >= residual_long["capacity_kwh"] * 0.10
residual_long["hour"] = residual_long["forecast_kst_dtm"].dt.hour
residual_long["month"] = residual_long["forecast_kst_dtm"].dt.month
residual_long["ficr_zone"] = np.select(
    [residual_long["abs_error_rate"] <= 0.06, residual_long["abs_error_rate"] <= 0.08],
    ["<=6%", "6-8%"],
    default=">8%",
)
residual_long.to_csv(RESIDUAL_LONG_PATH, index=False, encoding="utf-8-sig")

component_residual_long = build_component_residual_long(validation_predictions)
component_residual_long["forecast_kst_dtm"] = pd.to_datetime(component_residual_long["forecast_kst_dtm"])
component_residual_long["capacity_kwh"] = component_residual_long["group"].map(CAPACITY_KWH)
component_residual_long["residual_capacity_ratio"] = component_residual_long["residual_kwh"] / component_residual_long["capacity_kwh"]
component_residual_long["hour"] = component_residual_long["forecast_kst_dtm"].dt.hour
component_residual_long.to_csv(CHECKPOINT_DIR / "validation_component_residuals_long.csv", index=False, encoding="utf-8-sig")

summary = (
    residual_long
    .groupby("group")
    .agg(
        n_rows=("final_residual_kwh", "size"),
        eval_rows=("eval_mask", "sum"),
        residual_mean_kwh=("final_residual_kwh", "mean"),
        residual_median_kwh=("final_residual_kwh", "median"),
        residual_std_kwh=("final_residual_kwh", "std"),
        residual_skew_kwh=("final_residual_kwh", "skew"),
        mae_kwh=("abs_residual_kwh", "mean"),
        mean_abs_error_rate=("abs_error_rate", "mean"),
        ficr_fail_ratio=("ficr_zone", lambda s: float(np.mean(s == ">8%"))),
    )
    .reset_index()
)
display(summary)

if RUN_DIAGNOSTICS and PLOT_AVAILABLE:
    eval_df = residual_long[residual_long["eval_mask"]].copy()

    plt.figure(figsize=(12, 5))
    sns.kdeplot(data=residual_long, x="final_residual_kwh", hue="group", common_norm=False)
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Final residual KDE by group: prediction - actual, all rows")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_residual_kde_by_group_all_rows.png", dpi=160)
    plt.show()

    plt.figure(figsize=(12, 5))
    sns.kdeplot(data=eval_df, x="final_residual_kwh", hue="group", common_norm=False)
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Final residual KDE by group: prediction - actual, evaluated rows only")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_residual_kde_by_group_eval_rows.png", dpi=160)
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=residual_long, x="group", y="final_residual_kwh", showfliers=False)
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Final residual boxplot by group: prediction - actual, all rows")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_residual_boxplot_by_group_no_fliers.png", dpi=160)
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=eval_df, x="group", y="final_residual_kwh", showfliers=False)
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Final residual boxplot by group: prediction - actual, evaluated rows only")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_residual_boxplot_by_group_eval_rows_no_fliers.png", dpi=160)
    plt.show()

    plt.figure(figsize=(14, 5))
    sns.boxplot(data=eval_df, x="hour", y="final_residual_kwh", hue="group", showfliers=False)
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Final residual by hour: prediction - actual, evaluated rows only")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_residual_boxplot_by_hour_eval_rows.png", dpi=160)
    plt.show()

    for group in TARGET_COLS:
        sub = validation_predictions[validation_predictions["group"] == group].copy()
        if sub.empty:
            continue
        sub["forecast_kst_dtm"] = pd.to_datetime(sub["forecast_kst_dtm"])
        sub = sub.sort_values("forecast_kst_dtm")

        plt.figure(figsize=(18, 5))
        plt.plot(sub["forecast_kst_dtm"], sub["actual_kwh"], label="actual", linewidth=1.0, alpha=0.85)
        plt.plot(sub["forecast_kst_dtm"], sub["final_pred_kwh"], label="final prediction", linewidth=1.0, alpha=0.85)
        plt.title(f"Validation actual vs final prediction over time: {group}")
        plt.xlabel("forecast_kst_dtm")
        plt.ylabel("kWh")
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"validation_actual_vs_final_pred_timeseries_{group}.png", dpi=160)
        plt.show()

    plt.figure(figsize=(13, 5))
    sns.boxplot(data=component_residual_long, x="component", y="residual_kwh", hue="group", showfliers=False)
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Component residual boxplot: prediction - actual")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "component_residual_boxplot.png", dpi=160)
    plt.show()

    for group in TARGET_COLS:
        sub = component_residual_long[component_residual_long["group"] == group].copy()
        if sub.empty:
            continue
        plt.figure(figsize=(12, 5))
        sns.kdeplot(data=sub, x="residual_kwh", hue="component", common_norm=False)
        plt.axvline(0, color="black", linestyle="--", linewidth=1)
        plt.title(f"Component residual KDE: prediction - actual, {group}")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"component_residual_kde_{group}.png", dpi=160)
        plt.show()

    if len(wrong_regime_cases):
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=wrong_regime_cases, x="group", y="actual_kwh", hue="wrong_model", showfliers=False)
        plt.title("True generation for wrong Model I/II cases")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "wrong_regime_cases_true_generation_boxplot.png", dpi=160)
        plt.show()

    print("Saved residual plots to:", PLOT_DIR)
else:
    print("Diagnostics plotting skipped.")

### AutoGluon Leaderboard and Optional Feature Importance

In [ ]:

if "autogluon_leaderboard" in globals() and not autogluon_leaderboard.empty:
    display(autogluon_leaderboard.groupby(["group", "component"]).head(10))
    autogluon_leaderboard.to_csv(AUTOGLUON_LEADERBOARD_PATH, index=False, encoding="utf-8-sig")
else:
    print("AutoGluon leaderboard is unavailable. Run the AutoGluon training cell first.")


if RUN_AUTOGLUON_FEATURE_IMPORTANCE and "validation_artifacts" in globals():
    ag_importance_frames = []
    for group, artifact in validation_artifacts.items():
        for component, model_key in [
            ("model1_zero", "zero_model"),
            ("model2_capacity", "capacity_model"),
            ("model3_regressor", "reg_model"),
        ]:
            model = artifact[model_key]
            data = artifact["tuning_frames"].get(component)
            if isinstance(model, ConstantBinaryModel) or data is None:
                continue
            try:
                fi = model.feature_importance(
                    data=data,
                    num_shuffle_sets=PERMUTATION_N_REPEATS,
                    subsample_size=min(PERMUTATION_SAMPLE_SIZE, len(data)),
                    silent=True,
                )
                fi = fi.reset_index().rename(columns={"index": "feature"})
                fi.insert(0, "group", group)
                fi.insert(1, "component", component)
                ag_importance_frames.append(fi)
            except Exception as e:
                print(f"[WARN] AutoGluon feature importance failed for {group}/{component}: {e}")

    autogluon_feature_importance = (
        pd.concat(ag_importance_frames, ignore_index=True)
        if ag_importance_frames
        else pd.DataFrame()
    )
    if not autogluon_feature_importance.empty:
        path = IMPORTANCE_DIR / "autogluon_feature_importance.csv"
        autogluon_feature_importance.to_csv(path, index=False, encoding="utf-8-sig")
        display(autogluon_feature_importance.groupby(["group", "component"]).head(30))
        print("Saved AutoGluon feature importance:", path)
else:
    print("AutoGluon feature importance skipped. Set RUN_AUTOGLUON_FEATURE_IMPORTANCE = True to compute it.")


## (8) Final Prediction and Submission

The selected group-wise configurations are refit on all available labels and used to predict the test period.


In [ ]:

if RUN_FINAL_TRAIN_AND_SUBMIT:
    submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    final_summary_rows = []
    final_artifacts = {}
    final_leaderboard_frames = []

    for group in TARGET_COLS:
        print(f"\n===== AutoGluon final fit: {group} =====")
        artifact = fit_regime_pipeline(group, use_validation_fit=False, phase="final")
        selected = artifact["selected"]
        test_df = artifact["test_df"]
        X_test, _ = clean_matrix(test_df[selected], artifact["medians"])
        pred = predict_regime_pipeline(artifact, X_test, group)

        aligned = (
            pd.DataFrame({
                "forecast_id": test_df["forecast_id"],
                group: pred["final_pred"],
            })
            .set_index("forecast_id")
            .loc[submission["forecast_id"], group]
            .to_numpy()
        )
        submission[group] = np.clip(aligned, 0.0, CAPACITY_KWH[group])

        for component, model_key in [
            ("model1_zero", "zero_model"),
            ("model2_capacity", "capacity_model"),
            ("model3_regressor", "reg_model"),
        ]:
            final_leaderboard_frames.append(
                safe_autogluon_leaderboard(
                    artifact[model_key],
                    artifact["tuning_frames"].get(component),
                    group,
                    component,
                    "final",
                )
            )

        final_summary_rows.append({
            "group": group,
            "train_rows": int(artifact["train_mask"].sum()),
            "test_rows": len(test_df),
            "n_features": len(selected),
            "zero_fire_rate_test": float(np.mean(pred["zero_flag"])),
            "capacity_fire_rate_test": float(np.mean(pred["capacity_flag"])),
            "both_fire_rate_test": float(np.mean(pred["zero_flag"] & pred["capacity_flag"])),
            "prediction_min": float(np.min(submission[group])),
            "prediction_mean": float(np.mean(submission[group])),
            "prediction_max": float(np.max(submission[group])),
        })
        final_artifacts[group] = artifact
        print(group, final_summary_rows[-1])

    for target in TARGET_COLS:
        if target not in submission.columns:
            submission[target] = 0.0
        submission[target] = submission[target].clip(lower=0.0, upper=CAPACITY_KWH[target])

    submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
        submission[["forecast_id", *TARGET_COLS]],
        on="forecast_id",
        how="left",
    )
    submission[TARGET_COLS] = submission[TARGET_COLS].fillna(0.0)
    submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8-sig")

    final_summary = pd.DataFrame(final_summary_rows)
    final_summary.to_csv(FINAL_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    final_autogluon_leaderboard = (
        pd.concat([f for f in final_leaderboard_frames if f is not None and not f.empty], ignore_index=True)
        if final_leaderboard_frames
        else pd.DataFrame()
    )
    if not final_autogluon_leaderboard.empty:
        final_autogluon_leaderboard.to_csv(CHECKPOINT_DIR / "autogluon_final_leaderboard.csv", index=False, encoding="utf-8-sig")

    display(final_summary)
    display(submission[TARGET_COLS].agg(["min", "mean", "max"]).T)
    print("Saved submission:", SUBMISSION_PATH)
    print("Saved final summary:", FINAL_SUMMARY_PATH)
else:
    print("Final training and submission skipped.")


In [ ]:
print("Experiment directory:", EXP_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Plot directory:", PLOT_DIR)
print("Importance directory:", IMPORTANCE_DIR)
print("SHAP directory:", SHAP_DIR)
print("Submission path:", SUBMISSION_PATH)